# Dataset Loading

In [1]:
from datasets import load_dataset

ds = load_dataset("eQOURSE/jee-main-questions", data_dir="chemistry")

In [2]:
print(ds)

print(ds["train"].column_names)

print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['question_id', 'question', 'question_images', 'option_1', 'option_2', 'option_3', 'option_4', 'correct_option', 'numerical_answer', 'solution', 'solution_images', 'subject', 'topic', 'subtopic', 'difficulty', 'question_type', 'has_image', 'exam', 'source_paper'],
        num_rows: 627
    })
    test: Dataset({
        features: ['question_id', 'question', 'question_images', 'option_1', 'option_2', 'option_3', 'option_4', 'correct_option', 'numerical_answer', 'solution', 'solution_images', 'subject', 'topic', 'subtopic', 'difficulty', 'question_type', 'has_image', 'exam', 'source_paper'],
        num_rows: 111
    })
})
['question_id', 'question', 'question_images', 'option_1', 'option_2', 'option_3', 'option_4', 'correct_option', 'numerical_answer', 'solution', 'solution_images', 'subject', 'topic', 'subtopic', 'difficulty', 'question_type', 'has_image', 'exam', 'source_paper']
{'question_id': 'CH-24-Q25', 'question': 'In an experi

# Text preprocessing

In [2]:
from datasets import load_dataset
import pandas as pd
import numpy as np

# 1. LOAD DATASET

print("Loading dataset...")

ds = load_dataset(
    "eQOURSE/jee-main-questions",
    data_dir="chemistry"
)

df = ds["train"].to_pandas()

print("\nOriginal shape:", df.shape)
print("Original columns:")
print(df.columns.tolist())

# 2. FIND ORIGINAL QUESTION ID COLUMN
possible_id_columns = [
    "id",
    "question_id",
    "_id",
    "uuid",
    "ID",
    "questionId"
]

id_column = None

for col in possible_id_columns:
    if col in df.columns:
        id_column = col
        break

if id_column is None:
    raise ValueError(
        "No original question ID column found."
    )
print("\nOriginal ID column:", id_column)

# 3. CHECK REQUIRED COLUMNS
required_columns = [
    id_column,
    "question",
    "solution",
    "subject",
    "topic",
    "difficulty",
    "question_images"
]
missing_columns = [
    col for col in required_columns
    if col not in df.columns
]
if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )
print("\nAll required columns are available.")

# 4. TEXT-ONLY QUESTION FUNCTION
def is_text_only(value):

    if value is None:
        return True
    if isinstance(value, float) and np.isnan(value):
        return True
    if isinstance(value, (list, np.ndarray)):
        return len(value) == 0
    if isinstance(value, str):
        value = value.strip()
        return value in [
            "",
            "[]",
            "None",
            "null",
            "NULL",
            "nan",
            "NaN"
        ]
    return False

# 5. KEEP ONLY TEXT QUESTIONS
before = len(df)
df = df[
    df["question_images"].apply(is_text_only)
].copy()
print("\nTEXT-ONLY FILTER")
print("Before:", before)
print("After :", len(df))
print("Removed:", before - len(df))

# 6. REMOVE MISSING / EMPTY QUESTIONS
before = len(df)
df = df.dropna(
    subset=["question"]
).copy()
df["question"] = (
    df["question"]
    .astype(str)
    .str.strip()
)
df = df[
    df["question"] != ""
].copy()
print("\nEMPTY QUESTION FILTER")
print("Before:", before)
print("After :", len(df))
print("Removed:", before - len(df))

# 7. REMOVE DUPLICATE QUESTIONS
before = len(df)

df = df.drop_duplicates(
    subset=["question"],
    keep="first"
).copy()
print("\nDUPLICATE QUESTION FILTER")
print("Before:", before)
print("After :", len(df))
print("Removed:", before - len(df))

# 8. REMOVE DUPLICATE ORIGINAL IDs
before = len(df)
df = df.drop_duplicates(
    subset=[id_column],
    keep="first"
).copy()

print("\nDUPLICATE ID FILTER")
print("Before:", before)
print("After :", len(df))
print("Removed:", before - len(df))


# 9. CREATE FINAL DATAFRAME DIRECTLY
final_df = pd.DataFrame({
    "original_question_id": df[id_column].astype(str),
    "question": df["question"],
    "answer": df["solution"],
    "topic": df["subject"],
    "subtopic": df["topic"],
    "difficulty": df["difficulty"]
})

# 10. CLEAN FINAL COLUMNS
for col in final_df.columns:
 # Convert each column to a Series of strings
    final_df[col] = (
        final_df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# 11. REMOVE QUESTIONS WITH MISSING IDs
before = len(final_df)
final_df = final_df[
    final_df["original_question_id"] != ""
].copy()

print("\nMISSING ID FILTER")
print("Before:", before)
print("After :", len(final_df))
print("Removed:", before - len(final_df))

# 12. RESET INDEX
final_df = final_df.reset_index(drop=True)

# 13. FINAL VALIDATION
print("\n" + "=" * 70)
print("FINAL PREPROCESSED DATASET")
print("=" * 70)

print("\nFinal shape:")
print(final_df.shape)

print("\nFinal columns:")
print(final_df.columns.tolist())

print("\nTotal questions:")
print(len(final_df))

print("\nUnique original question IDs:")
print(
    final_df["original_question_id"].nunique()
)

print("\nDuplicate original IDs:")
print(
    final_df["original_question_id"].duplicated().sum()
)

print("\nMissing original IDs:")
print(
    (
        final_df["original_question_id"]
        .str.strip()
        .eq("")
    ).sum()
)

# 15. SAVE FINAL DATASET

output_file = "chemistry_text_questions_preprocessed.csv"
final_df.to_csv(
    output_file,
    index=False
)

# 16. FINAL SUMMARY
print("\n" + "=" * 70)
print("PREPROCESSING COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nSaved file:")
print(output_file)

print("\nFinal columns:")
print(final_df.columns.tolist())

print("\nFinal number of questions:")
print(len(final_df))

print("\nOriginal IDs preserved:")
print("YES")

print("\nImages removed:")
print("YES")

print("\nDuplicate questions removed:")
print("YES")

print("\nMissing questions removed:")
print("YES")

Loading dataset...

Original shape: (627, 19)
Original columns:
['question_id', 'question', 'question_images', 'option_1', 'option_2', 'option_3', 'option_4', 'correct_option', 'numerical_answer', 'solution', 'solution_images', 'subject', 'topic', 'subtopic', 'difficulty', 'question_type', 'has_image', 'exam', 'source_paper']

Original ID column: question_id

All required columns are available.

TEXT-ONLY FILTER
Before: 627
After : 492
Removed: 135

EMPTY QUESTION FILTER
Before: 492
After : 492
Removed: 0

DUPLICATE QUESTION FILTER
Before: 492
After : 486
Removed: 6

DUPLICATE ID FILTER
Before: 486
After : 484
Removed: 2

MISSING ID FILTER
Before: 484
After : 484
Removed: 0

FINAL PREPROCESSED DATASET

Final shape:
(484, 6)

Final columns:
['original_question_id', 'question', 'answer', 'topic', 'subtopic', 'difficulty']

Total questions:
484

Unique original question IDs:
484

Duplicate original IDs:
0

Missing original IDs:
0

PREPROCESSING COMPLETED SUCCESSFULLY

Saved file:
chemistr

# KNOWLEDGE GRAPH

In [3]:
# ============================================================
# INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q networkx pyvis datasets


# ============================================================
# IMPORTS
# ============================================================

from datasets import load_dataset
import networkx as nx
import pickle
import os

from pyvis.network import Network


# ============================================================
# LOAD DATASET
# ============================================================

ds = load_dataset(
    "eQOURSE/jee-main-questions",
    data_dir="chemistry"
)

print("Dataset loaded successfully!")
print(ds)


# ============================================================
# CREATE KNOWLEDGE GRAPH
# ============================================================

G = nx.DiGraph()

for split in ds:
    for i, item in enumerate(ds[split]):

        # -----------------------------
        # 1. Basic question information
        # -----------------------------

        qid = str(item["question_id"])
        question = str(item["question"])

        subject = str(item["subject"])
        topic = str(item["topic"])
        subtopic = str(item["subtopic"])
        difficulty = str(item["difficulty"])
        qtype = str(item["question_type"])

        # -----------------------------
        # 2. Create readable node IDs
        # -----------------------------

        subject_id = f"SUBJECT_{subject}"
        topic_id = f"TOPIC_{topic}"
        subtopic_id = f"SUBTOPIC_{subtopic}"
        question_id = f"QUESTION_{qid}"
        difficulty_id = f"DIFFICULTY_{difficulty}"

        # -----------------------------
        # 3. Add Subject
        # -----------------------------

        G.add_node(
            subject_id,
            type="Subject",
            label=subject
        )

        # -----------------------------
        # 4. Add Topic
        # -----------------------------

        G.add_node(
            topic_id,
            type="Topic",
            label=topic
        )

        # -----------------------------
        # 5. Add Subtopic
        # -----------------------------

        G.add_node(
            subtopic_id,
            type="Subtopic",
            label=subtopic
        )

        # -----------------------------
        # 6. Add Question
        # -----------------------------

        G.add_node(
            question_id,
            type="Question",
            label=qid,
            question=question,
            question_type=qtype,
            split=split
        )

        # -----------------------------
        # 7. Add Difficulty
        # -----------------------------

        G.add_node(
            difficulty_id,
            type="Difficulty",
            label=difficulty
        )

        # -----------------------------
        # 8. Create hierarchy
        # -----------------------------

        G.add_edge(
            subject_id,
            topic_id,
            relation="HAS_TOPIC"
        )

        G.add_edge(
            topic_id,
            subtopic_id,
            relation="HAS_SUBTOPIC"
        )

        G.add_edge(
            subtopic_id,
            question_id,
            relation="HAS_QUESTION"
        )

        G.add_edge(
            question_id,
            difficulty_id,
            relation="HAS_DIFFICULTY"
        )

        # ==================================================
        # 9. OPTIONS
        # ==================================================

        options = [
            ("A", item["option_1"]),
            ("B", item["option_2"]),
            ("C", item["option_3"]),
            ("D", item["option_4"])
        ]

        for letter, option_text in options:

            if option_text is None:
                continue

            option_text = str(option_text).strip()

            if option_text == "":
                continue

            option_id = f"OPTION_{qid}_{letter}"

            G.add_node(
                option_id,
                type="Option",
                label=f"Option {letter}",
                text=option_text
            )

            G.add_edge(
                question_id,
                option_id,
                relation="HAS_OPTION"
            )

            # Correct answer
            correct = str(
                item["correct_option"]
            ).strip().upper()

            if correct == letter:

                G.add_edge(
                    question_id,
                    option_id,
                    relation="HAS_CORRECT_OPTION"
                )

        # ==================================================
        # 10. QUESTION IMAGES
        # ==================================================

        image_paths = item["question_images"]

        if image_paths:

            for j, image_path in enumerate(image_paths):

                if not image_path:
                    continue

                image_name = os.path.basename(
                    str(image_path)
                )

                image_id = f"IMAGE_{qid}_{j+1}"

                local_image = (
                    f"chemistry_images/"
                    f"{split}_{i}_{j}"
                    f"{os.path.splitext(str(image_path))[1]}"
                )

                G.add_node(
                    image_id,
                    type="QuestionImage",
                    label=image_name,
                    path=local_image
                )

                G.add_edge(
                    question_id,
                    image_id,
                    relation="HAS_IMAGE"
                )


# ============================================================
# GRAPH INFORMATION
# ============================================================

print("\n==============================================")
print("KNOWLEDGE GRAPH CREATED")
print("==============================================")

print("Nodes :", G.number_of_nodes())
print("Edges :", G.number_of_edges())


# ============================================================
# SAVE NETWORKX GRAPH AS .PKL
# ============================================================

pkl_path = "/kaggle/working/chemistry_knowledge_graph.pkl"

with open(pkl_path, "wb") as f:
    pickle.dump(G, f)

print("\nPKL file saved:")
print(pkl_path)


# ============================================================
# CREATE PYVIS VISUALIZATION
# ============================================================

net = Network(
    height="850px",
    width="100%",
    directed=True,
    bgcolor="#ffffff",
    font_color="#222222"
)


# ============================================================
# PYVIS OPTIONS
# ============================================================
# IMPORTANT:
# Physics is disabled initially because 3090 nodes is large.
# This prevents the Colab browser from getting stuck.
# ============================================================

net.set_options("""
{
  "nodes": {
    "font": {
      "size": 14
    },
    "borderWidth": 1
  },

  "edges": {
    "arrows": {
      "to": {
        "enabled": true
      }
    },
    "smooth": {
      "enabled": false
    }
  },

  "physics": {
    "enabled": false
  },

  "interaction": {
    "hover": true,
    "navigationButtons": true,
    "keyboard": true,
    "zoomView": true,
    "dragView": true
  }
}
""")


# ============================================================
# ADD NODES
# ============================================================

for node, data in G.nodes(data=True):

    node_type = data.get("type", "")
    label = str(data.get("label", node))

    # -----------------------------
    # SUBJECT
    # -----------------------------

    if node_type == "Subject":

        net.add_node(
            node,
            label=label,
            title=f"Subject: {label}",
            color="#FF6B6B",
            size=35
        )

    # -----------------------------
    # TOPIC
    # -----------------------------

    elif node_type == "Topic":

        net.add_node(
            node,
            label=label,
            title=f"Topic: {label}",
            color="#4ECDC4",
            size=30
        )

    # -----------------------------
    # SUBTOPIC
    # -----------------------------

    elif node_type == "Subtopic":

        net.add_node(
            node,
            label=label,
            title=f"Subtopic: {label}",
            color="#45B7D1",
            size=25
        )

    # -----------------------------
    # QUESTION
    # -----------------------------

    elif node_type == "Question":

        question = data.get("question", "")
        qtype = data.get("question_type", "")

        title = (
            f"<b>Question ID:</b> {label}<br>"
            f"<b>Type:</b> {qtype}<br><br>"
            f"{question}"
        )

        net.add_node(
            node,
            label=label,
            title=title,
            color="#96CEB4",
            size=18
        )

    # -----------------------------
    # OPTION
    # -----------------------------

    elif node_type == "Option":

        option_text = data.get("text", "")

        title = (
            f"<b>{label}</b><br>"
            f"{option_text}"
        )

        net.add_node(
            node,
            label=label,
            title=title,
            color="#FFEAA7",
            size=15
        )

    # -----------------------------
    # DIFFICULTY
    # -----------------------------

    elif node_type == "Difficulty":

        net.add_node(
            node,
            label=label,
            title=f"Difficulty: {label}",
            color="#DDA0DD",
            size=20
        )

    # -----------------------------
    # QUESTION IMAGE
    # -----------------------------

    elif node_type == "QuestionImage":

        image_path = data.get("path", "")

        net.add_node(
            node,
            label=label,
            title=(
                f"Question Image: {label}<br>"
                f"Path: {image_path}"
            ),
            color="#A29BFE",
            size=20
        )

    # -----------------------------
    # OTHER
    # -----------------------------

    else:

        net.add_node(
            node,
            label=label,
            title=label
        )


# ============================================================
# ADD EDGES
# ============================================================

for source, target, data in G.edges(data=True):

    relation = data.get("relation", "")

    net.add_edge(
        source,
        target,
        label=relation,
        title=relation
    )


# ============================================================
# SAVE PYVIS GRAPH AS HTML
# ============================================================

html_path = "/kaggle/working/chemistry_knowledge_graph.html"

net.save_graph(html_path)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n==============================================")
print("CHEMISTRY KNOWLEDGE GRAPH READY")
print("==============================================")

print("Nodes :", G.number_of_nodes())
print("Edges :", G.number_of_edges())

print("\nFiles created:")

print("\n1. NetworkX Knowledge Graph:")
print(pkl_path)

print("\n2. Interactive HTML Visualization:")
print(html_path)

Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['question_id', 'question', 'question_images', 'option_1', 'option_2', 'option_3', 'option_4', 'correct_option', 'numerical_answer', 'solution', 'solution_images', 'subject', 'topic', 'subtopic', 'difficulty', 'question_type', 'has_image', 'exam', 'source_paper'],
        num_rows: 627
    })
    test: Dataset({
        features: ['question_id', 'question', 'question_images', 'option_1', 'option_2', 'option_3', 'option_4', 'correct_option', 'numerical_answer', 'solution', 'solution_images', 'subject', 'topic', 'subtopic', 'difficulty', 'question_type', 'has_image', 'exam', 'source_paper'],
        num_rows: 111
    })
})

KNOWLEDGE GRAPH CREATED
Nodes : 3090
Edges : 3833

PKL file saved:
/kaggle/working/chemistry_knowledge_graph.pkl

CHEMISTRY KNOWLEDGE GRAPH READY
Nodes : 3090
Edges : 3833

Files created:

1. NetworkX Knowledge Graph:
/kaggle/working/chemistry_knowledge_graph.pkl

2. Interactive HTML Vis

# Names of all topics present in dataset

In [4]:
topics = set()

for split in ds:
    for row in ds[split]:
        topic = row["topic"]

        if topic is not None and str(topic).strip():
            topics.add(str(topic).strip())

print("Total unique topics:", len(topics))
print("\nTopics:")
for topic in sorted(topics):
    print("-", topic)

Total unique topics: 31

Topics:
- Alcohol, Phenol and Ethers
- Amines
- Atomic Structure
- Carbonyl Compound (Aldehyde and Ketone)
- Chemical Bonding
- Chemical Kinetics
- Coordination Compounds
- Electrochemistry
- Equivalent Concept
- Eudiometry
- Halogen
- Hydrocarbon
- IUPAC Nomenclature
- IUPAC Nomenclature and Structural Isomerism
- IUPAC and Structural Isomerism
- Ideal Gas
- Inorganic Chemistry
- Isomerism
- Mole concept
- Organic Chemistry
- P-block
- P-block VA, VIA, VIIA elements
- Periodic table
- Physical Chemistry
- Real Gas
- Redox Reaction
- S-block
- Salt Analysis
- Stereoisomerism
- Sub
- Surface Chemistry


# AGENT 1: REQUIREMENT ANALYSIS

In [5]:
# ============================================================
# AGENT 1: REQUIREMENT ANALYSIS AGENT
# Chemistry Question Generation Project
# ============================================================

from datasets import load_dataset
import re
import json


# ============================================================
# 1. LOAD / COMBINE DATASET
# ============================================================

# Make sure 'ds' is already loaded before running this code.
#
# Example:
# ds = load_dataset("eQOURSE/jee-main-questions", data_dir="chemistry")

# Combine train and test datasets
all_data = ds["train"].to_list() + ds["test"].to_list()


# ============================================================
# 2. REQUIREMENT ANALYSIS AGENT
# ============================================================

class RequirementAnalysisAgent:

    def __init__(self, dataset):

        self.dataset = dataset

        # ----------------------------------------------------
        # Extract broad topics from the topic column
        # ----------------------------------------------------

        self.available_topics = sorted(
            set(
                str(row["topic"]).strip()
                for row in dataset
                if row["topic"] is not None
                and str(row["topic"]).strip() != ""
            ),
            key=len,
            reverse=True
        )

        # ----------------------------------------------------
        # Extract specific topics from the subtopic column
        # ----------------------------------------------------

        self.available_subtopics = sorted(
            set(
                str(row["subtopic"]).strip()
                for row in dataset
                if row["subtopic"] is not None
                and str(row["subtopic"]).strip() != ""
            ),
            key=len,
            reverse=True
        )


    # ========================================================
    # ANALYZE USER REQUEST
    # ========================================================

    def analyze(self, user_request):

        request = user_request.lower().strip()


        # ====================================================
        # 1. EXTRACT NUMBER
        # ====================================================

        number_match = re.search(
            r"\b(\d+)\s*(?:questions?|mcqs?)\b",
            request
        )

        if number_match:

            number = int(number_match.group(1))

        else:

            around_match = re.search(
                r"\b(?:around|about|approximately)\s*(\d+)\b",
                request
            )

            if around_match:

                number = int(around_match.group(1))

            else:

                number = None


        # ====================================================
        # 2. EXTRACT QUESTION TYPE
        # ====================================================

        if re.search(
            r"\b(mcq|mcqs|multiple choice|objective)\b",
            request
        ):

            question_type = "MCQ"

        elif re.search(
            r"\b(numerical|numericals|integer type)\b",
            request
        ):

            question_type = "Numerical"

        else:

            question_type = None


        # ====================================================
        # 3. EXTRACT DIFFICULTY
        # ====================================================

        if re.search(
            r"\b(hard|difficult|challenging|tough)\b",
            request
        ):

            difficulty = "Hard"

        elif re.search(
            r"\b(easy|simple|basic)\b",
            request
        ):

            difficulty = "Easy"

        elif re.search(
            r"\b(moderate|medium|average)\b",
            request
        ):

            difficulty = "Moderate"

        else:

            difficulty = None


        # ====================================================
        # 4. EXTRACT TOPIC
        # ====================================================

        topic = None


        # ----------------------------------------------------
        # FIRST: CHECK IF USER WANTS ALL TOPICS
        # ----------------------------------------------------

        if re.search(
            r"\b(all topics|all topic|every topic|every topics|"
            r"all chapters|every chapter|entire syllabus|"
            r"whole syllabus|full syllabus|all chemistry)\b",
            request
        ):

            topic = "ALL"


        # ----------------------------------------------------
        # OTHERWISE: SEARCH FOR A SPECIFIC BROAD TOPIC
        # ----------------------------------------------------

        else:

            for available_topic in self.available_topics:

                if available_topic.lower() in request:

                    topic = available_topic

                    break


            # ------------------------------------------------
            # If broad topic not found, search subtopics
            # ------------------------------------------------

            if topic is None:

                for available_subtopic in self.available_subtopics:

                    if available_subtopic.lower() in request:

                        topic = available_subtopic

                        break


        # ====================================================
        # 5. RETURN STRUCTURED REQUIREMENTS
        # ====================================================

        requirements = {

            "Topic": topic,

            "Question Type": question_type,

            "Number": number,

            "Difficulty": difficulty

        }

        return requirements


# ============================================================
# 3. INITIALIZE AGENT 1
# ============================================================

agent_1 = RequirementAnalysisAgent(all_data)

print("\nAgent 1 initialized successfully!")


# ============================================================
# 4. INTERACTIVE USER INPUT
# ============================================================

print("\n==============================================")
print("CHEMISTRY QUESTION GENERATION SYSTEM")
print("AGENT 1: REQUIREMENT ANALYSIS")
print("==============================================")


print("\nEnter your question-generation request.")


# ============================================================
# 5. TAKE ONE REQUEST
# ============================================================

user_request = input("\nEnter your request: ")


# ============================================================
# 6. VALIDATE INPUT
# ============================================================

if not user_request.strip():

    print("\nPlease enter a valid request.")


else:

    # ========================================================
    # ANALYZE REQUEST
    # ========================================================

    requirements = agent_1.analyze(
        user_request
    )


    # ========================================================
    # DISPLAY RESULT
    # ========================================================

    print("\n----------------------------------------------")
    print("STRUCTURED REQUIREMENTS")
    print("----------------------------------------------")


    print(
        json.dumps(
            requirements,
            indent=4
        )
    )


    print("----------------------------------------------")


Agent 1 initialized successfully!

CHEMISTRY QUESTION GENERATION SYSTEM
AGENT 1: REQUIREMENT ANALYSIS

Enter your question-generation request.



Enter your request:  I want 200 MCQ questions from all topics with medium difficulty.



----------------------------------------------
STRUCTURED REQUIREMENTS
----------------------------------------------
{
    "Topic": "ALL",
    "Question Type": "MCQ",
    "Number": 200,
    "Difficulty": "Moderate"
}
----------------------------------------------


# AGENT 2: KNOWLEDGE GRAPH RETRIEVAL

In [6]:
# ============================================================

# AGENT 2 — KNOWLEDGE GRAPH RETRIEVAL AGENT

# Chemistry Question Generation Project

# ============================================================





# ============================================================

# 1. INSTALL REQUIRED PACKAGE

# ============================================================



!pip install -q rapidfuzz





# ============================================================

# 2. IMPORTS

# ============================================================



import os

import re

import json

import pickle

import networkx as nx



from rapidfuzz import fuzz

from datasets import load_dataset





# ============================================================

# 3. CONFIGURATION

# ============================================================



KG_PATH = "/kaggle/working/chemistry_knowledge_graph.pkl"



OUTPUT_JSON = "/kaggle/working/kg_retrieved_context.json"



OUTPUT_PKL = "/kaggle/working/kg_retrieved_context.pkl"





# ============================================================

# 4. LOAD OR BUILD KNOWLEDGE GRAPH

# ============================================================



# The old KG could contain stale image paths and no usable correct-answer

# edges.  Rebuild automatically when that happens.

REBUILD_IF_STALE = True

IMAGE_ROOT = "/kaggle/working/chemistry_images"

os.makedirs(IMAGE_ROOT, exist_ok=True)





def _normalize_correct_option(value):

    """Convert common dataset encodings (A, 1, option_1, etc.) to A-D."""

    if value is None:

        return None

    text = str(value).strip().upper()

    if not text or text in {"NONE", "NAN", "NULL"}:

        return None

    if text in {"A", "B", "C", "D"}:

        return text

    m = re.search(r"(?:OPTION[_ ]*)?([1-4])\b", text)

    if m:

        return "ABCD"[int(m.group(1)) - 1]

    m = re.search(r"\b([ABCD])\b", text)

    return m.group(1) if m else None





def _as_list(value):

    if value is None:

        return []

    if isinstance(value, (list, tuple)):

        return list(value)

    if isinstance(value, str):

        text = value.strip()

        if not text:

            return []

        try:

            parsed = json.loads(text)

            if isinstance(parsed, list):

                return parsed

        except Exception:

            pass

        return [text]

    return [value]





def _copy_or_materialize_image(image_value, split, row_index, image_index):

    """Return a real local image path whenever the dataset exposes image data/path."""

    import shutil

    from pathlib import Path



    raw = image_value

    source_path = None

    raw_bytes = None



    if isinstance(raw, dict):

        source_path = raw.get("path") or raw.get("filename") or raw.get("file")

        raw_bytes = raw.get("bytes")

    else:

        source_path = str(raw) if raw is not None else None



    basename = os.path.basename(str(source_path or f"image_{row_index}_{image_index}.png"))

    ext = os.path.splitext(basename)[1] or ".png"

    target = os.path.join(IMAGE_ROOT, f"{split}_{row_index}_{image_index}{ext}")



    if raw_bytes:

        try:

            with open(target, "wb") as f:

                f.write(raw_bytes)

            return target

        except Exception:

            pass



    candidates = []

    if source_path:

        candidates.extend([

            source_path,

            os.path.join("/kaggle/working", source_path),

            os.path.join(IMAGE_ROOT, basename),

            os.path.join("/kaggle/working", "chemistry", source_path),

            os.path.join("/kaggle/working", "chemistry", "images", basename),

        ])



    # Search the Colab content/cache for the exact basename before downloading.

    for root in ["/kaggle/working", "/root/.cache/huggingface"]:

        if os.path.isdir(root) and basename:

            for walk_root, _, files in os.walk(root):

                if basename in files:

                    candidates.append(os.path.join(walk_root, basename))

                    break



    for candidate in candidates:

        try:

            if candidate and os.path.isfile(candidate):

                shutil.copy2(candidate, target)

                return target

        except Exception:

            continue



    # If the dataset entry is a repository-relative filename, try the HF repo.

    try:

        from huggingface_hub import hf_hub_download

        repo_id = "eQOURSE/jee-main-questions"

        rels = []

        if source_path:

            rels.extend([str(source_path).lstrip("/"),

                         f"chemistry/{str(source_path).lstrip('/')}",

                         f"chemistry/images/{basename}",

                         f"images/{basename}"])

        for rel in dict.fromkeys(rels):

            try:

                downloaded = hf_hub_download(repo_id=repo_id, filename=rel, repo_type="dataset")

                shutil.copy2(downloaded, target)

                return target

            except Exception:

                continue

    except Exception:

        pass



    return None





def _kg_is_stale(graph):

    question_nodes = [n for n,d in graph.nodes(data=True) if d.get("type") == "Question"]

    if not question_nodes:

        return True

    # Old graphs had no correct-option edge at all.

    has_correct = any(

        graph.get_edge_data(q, n, {}).get("relation") == "HAS_CORRECT_OPTION"

        for q in question_nodes

        for n in graph.successors(q)

    )

    if not has_correct:

        return True

    # If an image node exists, its stored path must resolve.

    for n,d in graph.nodes(data=True):

        if d.get("type") == "QuestionImage":

            path = d.get("path", "")

            if not os.path.exists(path) and not os.path.exists(os.path.join("/kaggle/working", path)):

                return True

    return False





need_build = not os.path.exists(KG_PATH)

if not need_build:

    print("Loading existing Knowledge Graph...")

    with open(KG_PATH, "rb") as f:

        G = pickle.load(f)

    print("Knowledge Graph loaded successfully.")

    print("Nodes :", G.number_of_nodes())

    print("Edges :", G.number_of_edges())

    if REBUILD_IF_STALE and _kg_is_stale(G):

        print("Existing KG is stale (missing correct-answer edges or resolvable images).")

        print("Rebuilding Knowledge Graph from the dataset...")

        need_build = True



if need_build:

    if "ds" not in globals():

        ds = load_dataset("eQOURSE/jee-main-questions", data_dir="chemistry")



    G = nx.DiGraph()



    for split in ds:

        for i, item in enumerate(ds[split]):

            qid = str(item["question_id"])

            question = str(item["question"])

            subject = str(item["subject"])

            topic = str(item["topic"])

            subtopic = str(item["subtopic"])

            difficulty = str(item["difficulty"])

            qtype = str(item["question_type"])

            correct = _normalize_correct_option(item.get("correct_option"))



            subject_id = f"SUBJECT_{subject}"

            topic_id = f"TOPIC_{topic}"

            subtopic_id = f"SUBTOPIC_{subtopic}"

            question_id = f"QUESTION_{qid}"

            difficulty_id = f"DIFFICULTY_{difficulty}"



            G.add_node(subject_id, type="Subject", label=subject)

            G.add_node(topic_id, type="Topic", label=topic)

            G.add_node(subtopic_id, type="Subtopic", label=subtopic)

            G.add_node(question_id, type="Question", label=qid,

                       question=question, question_type=qtype,

                       split=split, correct_option=correct)

            G.add_node(difficulty_id, type="Difficulty", label=difficulty)



            G.add_edge(subject_id, topic_id, relation="HAS_TOPIC")

            G.add_edge(topic_id, subtopic_id, relation="HAS_SUBTOPIC")

            G.add_edge(subtopic_id, question_id, relation="HAS_QUESTION")

            G.add_edge(question_id, difficulty_id, relation="HAS_DIFFICULTY")



            options = [

                ("A", item.get("option_1")),

                ("B", item.get("option_2")),

                ("C", item.get("option_3")),

                ("D", item.get("option_4")),

            ]

            for letter, option_text in options:

                if option_text is None:

                    continue

                option_text = str(option_text).strip()

                if not option_text:

                    continue

                option_id = f"OPTION_{qid}_{letter}"

                G.add_node(option_id, type="Option", label=f"Option {letter}", text=option_text)

                relation = "HAS_CORRECT_OPTION" if correct == letter else "HAS_OPTION"

                G.add_edge(question_id, option_id, relation=relation)



            image_values = _as_list(item.get("question_images"))

            for j, image_value in enumerate(image_values):

                if image_value in (None, ""):

                    continue

                local_path = _copy_or_materialize_image(image_value, split, i, j)

                if local_path:

                    image_id = f"IMAGE_{qid}_{j+1}"

                    G.add_node(image_id, type="QuestionImage",

                               label=os.path.basename(str(image_value.get("path", image_value)) if isinstance(image_value, dict) else str(image_value)),

                               path=local_path,

                               source_path=(image_value.get("path") if isinstance(image_value, dict) else str(image_value)))

                    G.add_edge(question_id, image_id, relation="HAS_IMAGE")



    with open(KG_PATH, "wb") as f:

        pickle.dump(G, f)



    print("Knowledge Graph created and saved.")

    print("Nodes :", G.number_of_nodes())

    print("Edges :", G.number_of_edges())

    print("Resolved images:", sum(1 for _,d in G.nodes(data=True) if d.get("type") == "QuestionImage" and os.path.exists(d.get("path", ""))))



# ============================================================

# 5. NORMALIZATION FUNCTIONS

# ============================================================



def normalize_text(value):



    if value is None:



        return ""





    value = str(value).strip().lower()



    value = value.replace("_", " ")



    value = value.replace("-", " ")



    value = re.sub(r"\s+", " ", value)





    return value





# ============================================================

# 6. FIND SUBJECT

# ============================================================



def find_subject(subject):



    if not subject:



        return None





    requested = normalize_text(subject)



    best_node = None



    best_score = 0





    for node, data in G.nodes(data=True):



        if data.get("type") != "Subject":



            continue





        label = normalize_text(

            data.get("label", "")

        )





        score = fuzz.token_set_ratio(

            requested,

            label

        )





        if score > best_score:



            best_score = score



            best_node = node





    if best_score >= 60:



        return best_node





    return None





# ============================================================

# 7. FIND TOPIC

# ============================================================



def find_topic(topic):



    if not topic:



        return None





    # --------------------------------------------------------

    # "ALL" is not an actual topic node.

    # It is handled separately by Agent 2.

    # --------------------------------------------------------



    if normalize_text(topic) == "all":



        return None





    requested = normalize_text(topic)



    best_node = None



    best_score = 0





    for node, data in G.nodes(data=True):



        if data.get("type") != "Topic":



            continue





        label = normalize_text(

            data.get("label", "")

        )





        score = fuzz.token_set_ratio(

            requested,

            label

        )





        if score > best_score:



            best_score = score



            best_node = node





    if best_score >= 60:



        return best_node





    return None





# ============================================================

# 8. FIND CONCEPT / SUBTOPIC

# ============================================================



def find_concepts(concept):



    if not concept:



        return []





    requested = normalize_text(concept)



    results = []





    for node, data in G.nodes(data=True):



        if data.get("type") != "Subtopic":



            continue





        label = normalize_text(

            data.get("label", "")

        )





        score = fuzz.token_set_ratio(

            requested,

            label

        )





        if score >= 60:



            results.append(

                (node, score)

            )





    results.sort(

        key=lambda x: x[1],

        reverse=True

    )





    return [

        node

        for node, score in results

    ]





# ============================================================

# 9. GET QUESTIONS FROM A NODE

# ============================================================



def retrieve_questions(node):



    if node is None:



        return []





    questions = []





    for neighbor in G.successors(node):



        data = G.nodes[neighbor]





        if data.get("type") == "Question":



            questions.append(neighbor)





    return questions





# ============================================================

# 10. GET QUESTION DIFFICULTY

# ============================================================



def get_question_difficulty(question_node):



    for neighbor in G.successors(question_node):



        data = G.nodes[neighbor]





        if data.get("type") == "Difficulty":



            return data.get("label")





    return None





# ============================================================

# 11. GET QUESTION TYPE

# ============================================================



def get_question_type(question_node):



    data = G.nodes[question_node]





    return data.get(

        "question_type",

        ""

    )





# ============================================================

# 12. GET QUESTION ID

# ============================================================



def get_question_id(question_node):



    data = G.nodes[question_node]





    return data.get(

        "label",

        question_node.replace(

            "QUESTION_",

            ""

        )

    )





# ============================================================

# 13. GET OPTIONS

# ============================================================



def get_options(question_node):



    options = []





    for neighbor in G.successors(question_node):



        data = G.nodes[neighbor]





        if data.get("type") != "Option":



            continue





        # ----------------------------------------------------

        # Extract A/B/C/D from label

        # ----------------------------------------------------



        label = str(

            data.get("label", "")

        ).strip()





        match = re.search(

            r"([A-D])$",

            label.upper()

        )





        if match:



            letter = match.group(1)



        else:



            letter = ""





        text = data.get(

            "text",

            ""

        )





        # ----------------------------------------------------

        # Check whether this is the correct option

        # ----------------------------------------------------



        edge_data = G.get_edge_data(

            question_node,

            neighbor,

            {}

        )





        relation = edge_data.get(

            "relation"

        )





        is_correct = (

            relation == "HAS_CORRECT_OPTION"

        )





        options.append(

            {

                "option": letter,

                "text": text,

                "is_correct": is_correct

            }

        )





    options.sort(

        key=lambda x: x["option"]

    )





    return options





# ============================================================

# 14. GET QUESTION IMAGES

# ============================================================



def get_images(question_node):



    images = []





    for neighbor in G.successors(question_node):



        data = G.nodes[neighbor]





        if data.get("type") != "QuestionImage":



            continue





        images.append(

            {

                "image": data.get(

                    "label",

                    ""

                ),

                "path": data.get(

                    "path",

                    ""

                )

            }

        )





    return images





# ============================================================

# 15. FILTER QUESTIONS

# ============================================================



def filter_questions(

    question_nodes,

    difficulty=None,

    question_type=None

):



    filtered = []





    requested_difficulty = normalize_text(

        difficulty

    )





    requested_type = normalize_text(

        question_type

    )





    # --------------------------------------------------------

    # Dataset's "single_correct" is treated as MCQ

    # --------------------------------------------------------



    mcq_types = [

        "mcq",

        "mcqs",

        "multiple choice",

        "multiple choice question",

        "single correct",

        "single correct question",

        "single_correct"

    ]





    for question_node in question_nodes:



        actual_difficulty = normalize_text(

            get_question_difficulty(

                question_node

            )

        )





        actual_type = normalize_text(

            get_question_type(

                question_node

            )

        )





        # ----------------------------------------------------

        # Difficulty filter

        # ----------------------------------------------------



        if requested_difficulty:



            if actual_difficulty != requested_difficulty:



                continue





        # ----------------------------------------------------

        # Question type filter

        # ----------------------------------------------------



        if requested_type:



            if requested_type in mcq_types:



                if actual_type not in mcq_types:



                    continue



            else:



                if actual_type != requested_type:



                    continue





        filtered.append(

            question_node

        )





    return filtered





# ============================================================

# 16. RANK QUESTIONS

# ============================================================



def rank_questions(

    question_nodes,

    requirements

):



    requested_topic = normalize_text(

        requirements.get("Topic")

    )





    requested_concept = normalize_text(

        requirements.get("Concept")

    )





    requested_difficulty = normalize_text(

        requirements.get("Difficulty")

    )





    requested_type = normalize_text(

        requirements.get("Type")

        or requirements.get("Question Type")

    )





    mcq_types = [

        "mcq",

        "mcqs",

        "multiple choice",

        "multiple choice question",

        "single correct",

        "single correct question",

        "single_correct"

    ]





    ranked = []





    for question_node in question_nodes:



        data = G.nodes[question_node]



        score = 0





        # ====================================================

        # TOPIC SIMILARITY

        # ====================================================

        #

        # IMPORTANT:

        # If Topic = ALL, we DO NOT calculate topic

        # similarity because "ALL" is not a real topic.

        # ====================================================



        if requested_topic != "all":



            parent_topics = []





            for parent in G.predecessors(

                question_node

            ):



                parent_data = G.nodes[parent]





                if parent_data.get(

                    "type"

                ) == "Subtopic":



                    for topic_parent in G.predecessors(

                        parent

                    ):



                        topic_data = G.nodes[

                            topic_parent

                        ]





                        if topic_data.get(

                            "type"

                        ) == "Topic":



                            parent_topics.append(

                                topic_data.get(

                                    "label",

                                    ""

                                )

                            )





            for topic_label in parent_topics:



                topic_score = fuzz.token_set_ratio(

                    requested_topic,

                    normalize_text(

                        topic_label

                    )

                )





                score += topic_score * 0.5





        # ====================================================

        # CONCEPT SIMILARITY

        # ====================================================



        if requested_concept:



            for parent in G.predecessors(

                question_node

            ):



                parent_data = G.nodes[parent]





                if parent_data.get(

                    "type"

                ) == "Subtopic":



                    concept_score = fuzz.token_set_ratio(

                        requested_concept,

                        normalize_text(

                            parent_data.get(

                                "label",

                                ""

                            )

                        )

                    )





                    score += concept_score * 0.5





        # ====================================================

        # DIFFICULTY

        # ====================================================



        actual_difficulty = normalize_text(

            get_question_difficulty(

                question_node

            )

        )





        if (

            requested_difficulty

            and actual_difficulty

            == requested_difficulty

        ):



            score += 30





        # ====================================================

        # QUESTION TYPE

        # ====================================================



        actual_type = normalize_text(

            get_question_type(

                question_node

            )

        )





        if requested_type in mcq_types:



            if actual_type in mcq_types:



                score += 30





        elif (

            requested_type

            and actual_type

            == requested_type

        ):



            score += 30





        ranked.append(

            (

                question_node,

                score

            )

        )





    ranked.sort(

        key=lambda x: x[1],

        reverse=True

    )





    return [

        node

        for node, score in ranked

    ]





# ============================================================

# 17. BUILD REFERENCE QUESTION

# ============================================================



def build_reference_question(

    question_node

):



    data = G.nodes[

        question_node

    ]





    question_id = get_question_id(

        question_node

    )





    question = data.get(

        "question",

        ""

    )





    question_type = get_question_type(

        question_node

    )





    difficulty = get_question_difficulty(

        question_node

    )





    options = get_options(

        question_node

    )





    images = get_images(

        question_node

    )





    # --------------------------------------------------------

    # Find topic, subtopic and subject

    # --------------------------------------------------------



    subtopic = None



    topic = None



    subject = None





    for parent in G.predecessors(

        question_node

    ):



        parent_data = G.nodes[parent]





        if parent_data.get(

            "type"

        ) == "Subtopic":



            subtopic = parent_data.get(

                "label"

            )





            for topic_parent in G.predecessors(

                parent

            ):



                topic_data = G.nodes[

                    topic_parent

                ]





                if topic_data.get(

                    "type"

                ) == "Topic":



                    topic = topic_data.get(

                        "label"

                    )





                    for subject_parent in G.predecessors(

                        topic_parent

                    ):



                        subject_data = G.nodes[

                            subject_parent

                        ]





                        if subject_data.get(

                            "type"

                        ) == "Subject":



                            subject = subject_data.get(

                                "label"

                            )





    # --------------------------------------------------------

    # Correct option

    # --------------------------------------------------------



    correct_option = None





    for option in options:



        if option.get(

            "is_correct"

        ):



            correct_option = option.get(

                "option"

            )



            break





    return {



        "question_id": question_id,



        "question": question,



        "subject": subject,



        "topic": topic,



        "subtopic": subtopic,



        "difficulty": difficulty,



        "question_type": question_type,



        "options": options,



        "correct_option": correct_option,



        "images": images

    }





# ============================================================

# 18. MAIN AGENT 2 FUNCTION

# ============================================================



def kg_retrieval_agent(

    requirements

):



    print("\n" + "=" * 70)



    print("AGENT 2 — KNOWLEDGE GRAPH RETRIEVAL")



    print("=" * 70)





    # ========================================================

    # GET REQUIREMENTS FROM AGENT 1

    # ========================================================



    number = requirements.get(

        "Number",

        5

    )





    try:



        number = int(number)



    except:



        number = 5





    exam = requirements.get(

        "Exam"

    ) or "JEE Main"





    subject = requirements.get(

        "Subject"

    ) or "Chemistry"





    topic = requirements.get(

        "Topic"

    )





    concept = requirements.get(

        "Concept"

    )





    difficulty = requirements.get(

        "Difficulty"

    )





    # Agent 1 uses "Question Type"

    question_type = (

        requirements.get("Question Type")

        or requirements.get("Type")

    )





    # ========================================================

    # NORMALIZE REQUIREMENTS

    # ========================================================



    normalized_requirements = {



        "Number": number,



        "Exam": exam,



        "Subject": subject,



        "Topic": topic,



        "Concept": concept,



        "Difficulty": difficulty,



        "Type": question_type

    }





    print(

        "\nRequirements received from Agent 1:"

    )





    print(

        json.dumps(

            normalized_requirements,

            indent=4

        )

    )





    # ========================================================

    # RETRIEVE QUESTIONS

    # ========================================================



    # --------------------------------------------------------

    # CASE 1: ALL TOPICS

    # --------------------------------------------------------



    if normalize_text(topic) == "all":



        print(

            "\nTopic requirement: ALL TOPICS"

        )



        print(

            "Retrieving questions from all topics..."

        )





        all_question_nodes = set()





        # ----------------------------------------------------

        # Traverse every Topic node

        # ----------------------------------------------------



        for topic_node in G.nodes:



            topic_data = G.nodes[

                topic_node

            ]





            if topic_data.get(

                "type"

            ) != "Topic":



                continue





            # ------------------------------------------------

            # Traverse subtopics

            # ------------------------------------------------



            for subtopic_node in G.successors(

                topic_node

            ):



                subtopic_data = G.nodes[

                    subtopic_node

                ]





                if subtopic_data.get(

                    "type"

                ) != "Subtopic":



                    continue





                # --------------------------------------------

                # Retrieve questions

                # --------------------------------------------



                subtopic_questions = retrieve_questions(

                    subtopic_node

                )





                all_question_nodes.update(

                    subtopic_questions

                )





        all_question_nodes = list(

            all_question_nodes

        )





        print(

            "Total questions across ALL topics:",

            len(all_question_nodes)

        )





    # --------------------------------------------------------

    # CASE 2: SPECIFIC TOPIC

    # --------------------------------------------------------



    else:



        topic_node = find_topic(

            topic

        )





        if topic_node is None:



            print(

                "\nTopic not found in Knowledge Graph."

            )





            return {



                "requirements":

                    normalized_requirements,



                "questions": [],



                "retrieved_count": 0,



                "fallback_used": False

            }





        print(

            "\nMatched Topic:",

            G.nodes[

                topic_node

            ].get(

                "label"

            )

        )





        # ----------------------------------------------------

        # Questions directly under topic

        # ----------------------------------------------------



        topic_questions = retrieve_questions(

            topic_node

        )





        print(

            "Questions directly under topic:",

            len(topic_questions)

        )





        # ----------------------------------------------------

        # Questions through subtopics

        # ----------------------------------------------------



        all_question_nodes = set(

            topic_questions

        )





        for subtopic_node in G.successors(

            topic_node

        ):



            subtopic_data = G.nodes[

                subtopic_node

            ]





            if subtopic_data.get(

                "type"

            ) != "Subtopic":



                continue





            subtopic_questions = retrieve_questions(

                subtopic_node

            )





            all_question_nodes.update(

                subtopic_questions

            )





        all_question_nodes = list(

            all_question_nodes

        )





        print(

            "Total topic questions:",

            len(all_question_nodes)

        )





    # ========================================================

    # CONCEPT FILTER

    # ========================================================



    if concept:



        concept_nodes = find_concepts(

            concept

        )





        concept_questions = []





        for concept_node in concept_nodes:



            concept_questions.extend(

                retrieve_questions(

                    concept_node

                )

            )





        concept_questions = list(

            set(

                concept_questions

            )

        )





        if concept_questions:



            all_question_nodes = concept_questions





            print(

                "Questions after concept matching:",

                len(all_question_nodes)

            )





    # ========================================================

    # EXACT FILTERING

    # ========================================================



    filtered_questions = filter_questions(

        all_question_nodes,

        difficulty=difficulty,

        question_type=question_type

    )





    print(

        "Questions after exact filtering:",

        len(filtered_questions)

    )





    # ========================================================

    # FALLBACK

    # ========================================================



    fallback_used = False





    if len(filtered_questions) < number:



        fallback_used = True





        print(

            "\nExact filtering did not provide enough questions."

        )





        print(

            "Using ranked questions as fallback..."

        )





        # ----------------------------------------------------

        # Important:

        # If ALL topics were requested, rank all retrieved

        # questions without topic preference.

        # ----------------------------------------------------



        ranked_questions = rank_questions(

            all_question_nodes,

            normalized_requirements

        )





    else:



        ranked_questions = rank_questions(

            filtered_questions,

            normalized_requirements

        )





    # ========================================================

    # SELECT REQUESTED NUMBER

    # ========================================================



    selected_questions = ranked_questions[

        :number

    ]





    # ========================================================

    # BUILD FINAL REFERENCES

    # ========================================================



    references = []





    for question_node in selected_questions:



        reference = build_reference_question(

            question_node

        )





        references.append(

            reference

        )





    # ========================================================

    # FINAL RESULT

    # ========================================================



    result = {



        "requirements":

            normalized_requirements,



        "retrieved_count":

            len(references),



        "fallback_used":

            fallback_used,



        "questions":

            references

    }





    # ========================================================

    # SAVE JSON

    # ========================================================



    with open(

        OUTPUT_JSON,

        "w",

        encoding="utf-8"

    ) as f:



        json.dump(

            result,

            f,

            indent=4,

            ensure_ascii=False

        )





    # ========================================================

    # SAVE PKL

    # ========================================================



    with open(

        OUTPUT_PKL,

        "wb"

    ) as f:



        pickle.dump(

            result,

            f

        )





    # ========================================================

    # SUMMARY

    # ========================================================



    print("\n" + "=" * 70)



    print("AGENT 2 COMPLETED")



    print("=" * 70)





    print(

        "Requested questions :",

        number

    )





    print(

        "Questions retrieved :",

        len(references)

    )





    print(

        "Fallback used       :",

        fallback_used

    )





    print(

        "JSON saved to       :",

        OUTPUT_JSON

    )





    print(

        "PKL saved to        :",

        OUTPUT_PKL

    )





    return result





# ============================================================

# 19. NORMAL MODE

# AGENT 2 TAKES THE OUTPUT OF AGENT 1

# ============================================================



# Agent 1 creates:

#

# requirements = agent_1.analyze(user_request)

#

# Agent 2 receives those requirements.





agent2_result = kg_retrieval_agent(

    requirements

)





# ============================================================

# 20. DISPLAY RETRIEVED QUESTIONS

# ============================================================



# print("\n")



# print("=" * 70)



# print("RETRIEVED QUESTIONS")



# print("=" * 70)





# for i, question in enumerate(

#     agent2_result["questions"],

#     start=1

# ):



#     print(

#         f"\nQuestion {i}"

#     )





#     print(

#         "ID:",

#         question["question_id"]

#     )





#     print(

#         "Question:",

#         question["question"]

#     )





#     print(

#         "Topic:",

#         question["topic"]

#     )





#     print(

#         "Subtopic:",

#         question["subtopic"]

#     )





#     print(

#         "Difficulty:",

#         question["difficulty"]

#     )





#     print(

#         "Type:",

#         question["question_type"]

#     )





#     print(

#         "Options:",

#         question["options"]

#     )





#     print(

#         "Correct:",

#         question["correct_option"]

#     )





#     print(

#         "Images:",

#         question["images"]

#     )

Loading existing Knowledge Graph...
Knowledge Graph loaded successfully.
Nodes : 3090
Edges : 3833
Existing KG is stale (missing correct-answer edges or resolvable images).
Rebuilding Knowledge Graph from the dataset...
Knowledge Graph created and saved.
Nodes : 3090
Edges : 3833
Resolved images: 299

AGENT 2 — KNOWLEDGE GRAPH RETRIEVAL

Requirements received from Agent 1:
{
    "Number": 200,
    "Exam": "JEE Main",
    "Subject": "Chemistry",
    "Topic": "ALL",
    "Concept": null,
    "Difficulty": "Moderate",
    "Type": "MCQ"
}

Topic requirement: ALL TOPICS
Retrieving questions from all topics...
Total questions across ALL topics: 731
Questions after exact filtering: 325

AGENT 2 COMPLETED
Requested questions : 200
Questions retrieved : 200
Fallback used       : False
JSON saved to       : /kaggle/working/kg_retrieved_context.json
PKL saved to        : /kaggle/working/kg_retrieved_context.pkl


# AGENT 3: PROMPT CONSTRUCTION AGENT

In [7]:
# ============================================================

# AGENT 3: PROMPT CONSTRUCTION AGENT

# Chemistry Question Generation Project

#

# INPUT:

#   1. Agent 1 requirements

#   2. Agent 2 retrieved KG context

#

# OUTPUT:

#   Final structured prompt for Question Generation Agent

#

# Agent 3 DOES NOT generate questions.

# It only constructs the generation prompt.

# ============================================================





import json

import pickle

from pathlib import Path





# ============================================================

# 1. PROMPT CONSTRUCTION AGENT

# ============================================================



class PromptConstructionAgent:



    def __init__(self):



        print("Prompt Construction Agent initialized!")





    # ========================================================

    # 2. BUILD REQUIREMENT SECTION

    # ========================================================



    def build_requirements(self, requirements):



        number = requirements.get(

            "Number"

        )



        exam = requirements.get(

            "Exam",

            "JEE Main"

        )



        subject = requirements.get(

            "Subject",

            "Chemistry"

        )



        topic = requirements.get(

            "Topic"

        )



        concept = requirements.get(

            "Concept",

            requirements.get("Subtopic")

        )



        difficulty = requirements.get(

            "Difficulty"

        )



        # Support both Agent 1 and Agent 2 formats

        question_type = requirements.get(

            "Type",

            requirements.get("Question Type")

        )



        requirement_text = f"""

TARGET REQUIREMENTS



Number of questions: {number}

Exam: {exam}

Subject: {subject}

Topic: {topic}

Concept: {concept}

Difficulty: {difficulty}

Question type: {question_type}

"""



        return requirement_text.strip()





    # ========================================================

    # 3. BUILD KG REFERENCE SECTION

    # ========================================================



    def build_reference_section(self, retrieved_context):



        if retrieved_context is None:



            retrieved_context = {}





        # ----------------------------------------------------

        # Agent 2 stores retrieved questions under "questions"

        # ----------------------------------------------------



        references = retrieved_context.get(

            "questions",

            []

        )





        if not references:



            return """

KNOWLEDGE GRAPH REFERENCES



No reference questions were retrieved.



Generate questions using the specified requirements

and Chemistry knowledge.

""".strip()





        reference_text = (

            "KNOWLEDGE GRAPH REFERENCES\n\n"

        )





        for i, reference in enumerate(

            references,

            start=1

        ):



            reference_text += (

                f"REFERENCE QUESTION {i}\n"

            )





            reference_text += (

                f"Question ID: "

                f"{reference.get('question_id', 'N/A')}\n"

            )





            reference_text += (

                f"Subject: "

                f"{reference.get('subject', 'N/A')}\n"

            )





            reference_text += (

                f"Topic: "

                f"{reference.get('topic', 'N/A')}\n"

            )





            reference_text += (

                f"Concept: "

                f"{reference.get('subtopic', 'N/A')}\n"

            )





            reference_text += (

                f"Difficulty: "

                f"{reference.get('difficulty', 'N/A')}\n"

            )





            reference_text += (

                f"Question Type: "

                f"{reference.get('question_type', 'N/A')}\n"

            )





            reference_text += (

                f"Question:\n"

                f"{reference.get('question', 'N/A')}\n"

            )





            # ------------------------------------------------

            # Options

            # ------------------------------------------------



            options = reference.get(

                "options",

                []

            )





            if options:



                reference_text += "Options:\n"





                # Agent 2 returns:

                #

                # [

                #   {

                #       "option": "A",

                #       "text": "...",

                #       "is_correct": False

                #   }

                # ]

                #



                for option in options:



                    if isinstance(

                        option,

                        dict

                    ):



                        letter = option.get(

                            "option",

                            ""

                        )



                        text = option.get(

                            "text",

                            ""

                        )



                        reference_text += (

                            f"{letter}: {text}\n"

                        )



                    else:



                        reference_text += (

                            f"{option}\n"

                        )





            # ------------------------------------------------

            # Images

            # ------------------------------------------------



            question_images = reference.get(

                "images",

                []

            )





            if question_images:



                reference_text += (

                    "Question Images:\n"

                )





                for image in question_images:



                    if isinstance(

                        image,

                        dict

                    ):



                        image_path = image.get(

                            "path",

                            image.get(

                                "image",

                                ""

                            )

                        )



                    else:



                        image_path = image





                    reference_text += (

                        f"- {image_path}\n"

                    )





            # ------------------------------------------------

            # Correct answer

            # ------------------------------------------------



            correct_answer = reference.get(

                "correct_option"

            )





            if correct_answer:



                reference_text += (

                    f"Correct Answer: "

                    f"{correct_answer}\n"

                )





            reference_text += (

                "\n"

                + "-" * 60

                + "\n\n"

            )





        return reference_text.strip()





    # ========================================================

    # 4. BUILD GENERATION INSTRUCTIONS

    # ========================================================



    def build_instructions(self, requirements):



        question_type = requirements.get(

            "Type",

            requirements.get("Question Type")

        )





        instructions = """

GENERATION INSTRUCTIONS



Generate new Chemistry questions based on the target

requirements and the retrieved Knowledge Graph references.



The retrieved questions are reference material only.

DO NOT copy any reference question directly.



Create ORIGINAL questions while preserving the relevant

topic, concept, difficulty level and examination style.



The generated question must be chemically meaningful,

clear and unambiguous.



Ensure that the correct answer is supported by the

question and options.



Do not introduce information unrelated to the requested

topic or concept.

"""





        # Normalize question type

        normalized_type = str(

            question_type

        ).strip().lower()





        # ----------------------------------------------------

        # MCQ instructions

        # ----------------------------------------------------



        if normalized_type in [

            "mcq",

            "mcqs",

            "multiple choice",

            "multiple choice question",

            "objective",

            "single correct",

            "single_correct"

        ]:



            instructions += """

MCQ FORMAT



For every question provide:



1. Question

2. Option A

3. Option B

4. Option C

5. Option D

6. Correct Answer

7. Explanation



There must be exactly four options.



There must be exactly one correct option.



The distractors should be plausible and relevant.

"""





        # ----------------------------------------------------

        # Numerical instructions

        # ----------------------------------------------------



        elif normalized_type in [

            "numerical",

            "numericals",

            "integer type",

            "integer"

        ]:



            instructions += """

NUMERICAL FORMAT



For every question provide:



1. Question

2. Numerical Answer

3. Explanation



The numerical problem must be solvable from the

information provided.



Show the necessary reasoning in the explanation.

"""





        # ----------------------------------------------------

        # Fill in the blank instructions

        # ----------------------------------------------------



        elif normalized_type in [

            "fill in the blank",

            "fill-in-the-blank",

            "fill blank"

        ]:



            instructions += """

FILL IN THE BLANK FORMAT



For every question provide:



1. Question containing a blank

2. Correct Answer

3. Explanation



The blank must have one clear and chemically correct answer.

"""





        # ----------------------------------------------------

        # True or False instructions

        # ----------------------------------------------------



        elif normalized_type in [

            "true or false",

            "true/false"

        ]:



            instructions += """

TRUE OR FALSE FORMAT



For every question provide:



1. Statement

2. Answer: True or False

3. Explanation

"""





        # ----------------------------------------------------

        # Short answer instructions

        # ----------------------------------------------------



        elif normalized_type in [

            "short answer",

            "short-answer"

        ]:



            instructions += """

SHORT ANSWER FORMAT



For every question provide:



1. Question

2. Answer

3. Explanation

"""





        else:



            instructions += """

QUESTION TYPE



The requested question type was not explicitly specified.



Follow the available requirements and retrieved examples,

but do not assume a type that conflicts with the request.

"""





        return instructions.strip()





    # ========================================================

    # 5. BUILD OUTPUT FORMAT

    # ========================================================



    def build_output_format(self, requirements):



        question_type = requirements.get(

            "Type",

            requirements.get("Question Type")

        )





        normalized_type = str(

            question_type

        ).strip().lower()





        # ----------------------------------------------------

        # MCQ

        # ----------------------------------------------------



        if normalized_type in [

            "mcq",

            "mcqs",

            "multiple choice",

            "multiple choice question",

            "objective",

            "single correct",

            "single_correct"

        ]:



            return """

OUTPUT FORMAT



Return ONLY valid JSON.



Use this structure:



{

    "questions": [

        {

            "question": "...",

            "option_A": "...",

            "option_B": "...",

            "option_C": "...",

            "option_D": "...",

            "correct_answer": "A",

            "explanation": "..."

        }

    ]

}

""".strip()





        # ----------------------------------------------------

        # Numerical

        # ----------------------------------------------------



        elif normalized_type in [

            "numerical",

            "numericals",

            "integer type",

            "integer"

        ]:



            return """

OUTPUT FORMAT



Return ONLY valid JSON.



Use this structure:



{

    "questions": [

        {

            "question": "...",

            "numerical_answer": "...",

            "explanation": "..."

        }

    ]

}

""".strip()





        # ----------------------------------------------------

        # Fill in the blank

        # ----------------------------------------------------



        elif normalized_type in [

            "fill in the blank",

            "fill-in-the-blank",

            "fill blank"

        ]:



            return """

OUTPUT FORMAT



Return ONLY valid JSON.



Use this structure:



{

    "questions": [

        {

            "question": "...",

            "correct_answer": "...",

            "explanation": "..."

        }

    ]

}

""".strip()





        # ----------------------------------------------------

        # True / False

        # ----------------------------------------------------



        elif normalized_type in [

            "true or false",

            "true/false"

        ]:



            return """

OUTPUT FORMAT



Return ONLY valid JSON.



Use this structure:



{

    "questions": [

        {

            "statement": "...",

            "answer": "True",

            "explanation": "..."

        }

    ]

}

""".strip()





        # ----------------------------------------------------

        # Short answer

        # ----------------------------------------------------



        elif normalized_type in [

            "short answer",

            "short-answer"

        ]:



            return """

OUTPUT FORMAT



Return ONLY valid JSON.



Use this structure:



{

    "questions": [

        {

            "question": "...",

            "answer": "...",

            "explanation": "..."

        }

    ]

}

""".strip()





        # ----------------------------------------------------

        # Default

        # ----------------------------------------------------



        return """

OUTPUT FORMAT



Return ONLY valid JSON.



Use this structure:



{

    "questions": [

        {

            "question": "...",

            "answer": "...",

            "explanation": "..."

        }

    ]

}

""".strip()





    # ========================================================

    # 6. MAIN PROMPT BUILDER

    # ========================================================



    def build_prompt(

        self,

        requirements,

        retrieved_context

    ):



        requirement_section = (

            self.build_requirements(

                requirements

            )

        )





        reference_section = (

            self.build_reference_section(

                retrieved_context

            )

        )





        instruction_section = (

            self.build_instructions(

                requirements

            )

        )





        output_section = (

            self.build_output_format(

                requirements

            )

        )





        final_prompt = f"""

You are a Chemistry question generation system

specialized in JEE/NEET-style examination questions.



============================================================

{requirement_section}

============================================================



{reference_section}



============================================================

{instruction_section}

============================================================



============================================================

{output_section}

============================================================

""".strip()





        return final_prompt





# ============================================================

# 7. INITIALIZE AGENT 3

# ============================================================



agent_3 = PromptConstructionAgent()





# ============================================================

# 8. GET AGENT 1 OUTPUT

# ============================================================



# Agent 1 creates:

#

# requirements = agent_1.analyze(user_request)

#

# Agent 3 directly uses that variable.

#

# No new hard-coded requirements are used here.





if "requirements" not in globals():



    raise NameError(

        "Agent 1 requirements were not found. "

        "Run the Agent 1 cell first."

    )





# ============================================================

# 9. GET AGENT 2 OUTPUT

# ============================================================



# Agent 2 creates:

#

# agent2_result = kg_retrieval_agent(requirements)

#

# Agent 3 directly uses that result.





if "agent2_result" not in globals():



    raise NameError(

        "Agent 2 result was not found. "

        "Run the Agent 2 cell first."

    )





retrieved_context = agent2_result





# ============================================================

# 10. BUILD FINAL PROMPT

# ============================================================



final_prompt = agent_3.build_prompt(

    requirements,

    retrieved_context

)





# ============================================================

# 11. DISPLAY PROMPT

# ============================================================



print("\n")

print("=" * 70)

print("AGENT 3: PROMPT CONSTRUCTION")

print("=" * 70)



print("\nFINAL PROMPT:\n")

# print(final_prompt)





# ============================================================

# 12. SAVE PROMPT

# ============================================================



output_dir = Path("/kaggle/working")



output_dir.mkdir(

    parents=True,

    exist_ok=True

)





prompt_path = (

    output_dir

    / "question_generation_prompt.txt"

)





with open(

    prompt_path,

    "w",

    encoding="utf-8"

) as f:



    f.write(

        final_prompt

    )





# ============================================================

# 13. SAVE AS JSON

# ============================================================



prompt_json_path = (

    output_dir

    / "question_generation_prompt.json"

)





prompt_data = {

    "requirements": requirements,

    "retrieved_context": retrieved_context,

    "generation_prompt": final_prompt

}





with open(

    prompt_json_path,

    "w",

    encoding="utf-8"

) as f:



    json.dump(

        prompt_data,

        f,

        indent=4,

        ensure_ascii=False,

        default=str

    )





# ============================================================

# 14. COMPLETION MESSAGE

# ============================================================



print("\n")

print("=" * 70)

print("PROMPT CONSTRUCTION COMPLETED")

print("=" * 70)



print("\nPrompt saved to:")

print(prompt_path)



print("\nJSON prompt saved to:")

print(prompt_json_path)

Prompt Construction Agent initialized!


AGENT 3: PROMPT CONSTRUCTION

FINAL PROMPT:



PROMPT CONSTRUCTION COMPLETED

Prompt saved to:
/kaggle/working/question_generation_prompt.txt

JSON prompt saved to:
/kaggle/working/question_generation_prompt.json


# AGENT 4: MULTIMODAL CHEMISTRY QUESTION GENERATION

In [8]:
!pip uninstall -y transformers
!pip install --no-cache-dir transformers==4.51.3

Found existing installation: transformers 5.19.0
Uninstalling transformers-5.19.0:
  Successfully uninstalled transformers-5.19.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 86.4 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 334.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 184.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.2
    Uninstalling tokenizers-0.23.2:
      Successfully uninstalled tokenizers-0.23.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.

# Imports of agent 4

In [9]:
# ============================================================
# AGENT 4 — INSTALL + IMPORT EVERYTHING
# Run this cell once after starting/restarting Colab
# ============================================================

# ---------- Install required packages ----------
!pip install -q -U \
    "transformers>=4.51.0" \
    accelerate \
    bitsandbytes \
    rdkit \
    matplotlib \
    "python-docx>=1.1.2" \
    pillow


# ---------- Standard Python libraries ----------
import os
import re
import json
import time
import gc
import math
import html
import pickle
import base64
import random
import difflib
import traceback

from collections import Counter


# ---------- PIL ----------
from PIL import Image, ImageOps, ImageFilter


# ---------- PyTorch ----------
import torch


# ---------- Hugging Face ----------
import transformers
import accelerate
import bitsandbytes

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)


# ---------- RDKit ----------
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, rdMolDescriptors


# ---------- Matplotlib ----------
import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt


# ---------- Disable RDKit warnings ----------
RDLogger.DisableLog("rdApp.*")


# ---------- CUDA memory setting ----------
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# ============================================================
# VERSION CHECK
# ============================================================

print("=" * 60)
print("AGENT 4 DEPENDENCY CHECK")
print("=" * 60)

print("PyTorch       :", torch.__version__)
print("Transformers  :", transformers.__version__)
print("Accelerate    :", accelerate.__version__)
print("BitsAndBytes  :", bitsandbytes.__version__)

try:
    print("RDKit         :", Chem.rdBase.rdkitVersion)
except:
    print("RDKit         : OK")

print("Matplotlib    :", matplotlib.__version__)

try:
    import PIL
    print("Pillow        :", PIL.__version__)
except:
    print("Pillow        : OK")

print("=" * 60)


# ============================================================
# GPU CHECK
# ============================================================

if torch.cuda.is_available():

    print("GPU           :", torch.cuda.get_device_name(0))
    print("CUDA          :", torch.version.cuda)

else:

    print("WARNING: CUDA GPU is NOT available.")


print("=" * 60)
print("ALL IMPORTS SUCCESSFUL")
print("=" * 60)

AGENT 4 DEPENDENCY CHECK
PyTorch       : 2.11.0+cu128
Transformers  : 5.19.0
Accelerate    : 1.15.0
BitsAndBytes  : 0.50.2
RDKit         : 2026.03.6
Matplotlib    : 3.11.2
Pillow        : 12.3.0
GPU           : Tesla T4
CUDA          : 12.8
ALL IMPORTS SUCCESSFUL


# Agent 4 QUESTION GENERATION BOTH IMAGE AND TEXT

In [10]:
# ============================================================
# AGENT 4 v3: MULTIMODAL CHEMISTRY QUESTION GENERATION - MODERATE/HARD JEE MAIN
# Tesla T4 / Colab, Qwen3-8B (4-bit)
#
# WHAT'S NEW IN v2
#   IMAGE QUESTIONS (never fail, never turn into text, always correct)
#     * Built by CODE, not by the model:
#         - molecules from a bank of known-valid structures, drawn with RDKit
#         - answers COMPUTED by RDKit (sp2/sp3 carbons, chiral centres,
#           sigma/pi bonds, rings, formula, functional groups,
#           hybridisation / shape / lone pairs of central atoms)
#         - graphs drawn with matplotlib (first-order, zero-order,
#           exponential decay, Arrhenius plots) with computed answers
#     * Image and answer come from the same data -> they always match
#     * If one build fails, another molecule/graph is tried automatically
#
#   TEXT QUESTIONS (Knowledge Graph source -> Qwen3-8B -> checked)
#     * Source questions are retrieved from the Chemistry Knowledge Graph
#     * KG image questions are excluded from the text path
#     * Qwen3-8B generates a NEW question from each KG source
#     * Independent self-check, duplicate detection and structural validation
#     * Answer letters balanced by shuffling when safe
##
# RUN ORDER (after any runtime restart):
#   install cell -> restart -> Agent 1 -> Agent 2 -> Agent 3 -> THIS
# ============================================================


# ############################################################
# CELL 1  --  INSTALL  (run once on a fresh runtime, then RESTART)
# ############################################################
"""
!pip uninstall -y torchvision
!pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes rdkit matplotlib "python-docx>=1.1.2"
"""
# (Copy the two lines above, WITHOUT the triple quotes, into their own cell.)


# ############################################################
# CELL 2  --  MAIN SCRIPT
# ############################################################

import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import json
import time
import gc
import math
import html
import pickle
import base64
import random
import difflib
import traceback
from collections import Counter

import torch
import transformers
import accelerate
import bitsandbytes

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, rdMolDescriptors

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RDLogger.DisableLog("rdApp.*")

print("Transformers :", transformers.__version__)
print("Accelerate   :", accelerate.__version__)
print("BitsAndBytes :", bitsandbytes.__version__)
print("PyTorch      :", torch.__version__)


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = "Qwen/Qwen3-8B"

PROMPT_FILE = "/kaggle/working/question_generation_prompt.txt"
OUTPUT_FILE = "/kaggle/working/generated_chemistry_questions.json"
PICKLE_OUTPUT_FILE = "/kaggle/working/generated_chemistry_questions.pkl"
HTML_OUTPUT_FILE = "/kaggle/working/generated_chemistry_questions.html"
WORD_OUTPUT_FILE = "/kaggle/working/generated_chemistry_questions.docx"
IMAGE_OUTPUT_FOLDER = "/kaggle/working/generated_question_images"

os.makedirs(IMAGE_OUTPUT_FOLDER, exist_ok=True)

BATCH_SIZE = 5
MAX_BATCH_ATTEMPTS = 8            # attempts per batch for TEXT questions
MAX_NEW_TOKENS = 2200
TEMPERATURE = 0.70
RETRY_TEMPERATURE = 0.55
TOP_P = 0.90
TOP_K = 40
REPETITION_PENALTY = 1.05
MAX_AGENT3_CONTEXT_TOKENS = 1500

# --- self-check (verification) of model-written text questions ---
VERIFY_TEXT = True                # set False to skip (faster, less reliable)
VERIFY_MAX_TOKENS = 600
STRICT_VERIFY_ATTEMPTS = 5        # up to this attempt, disputed questions are rejected;
                                  # afterwards they are kept but flagged "disputed"

DUP_SEQ_RATIO = 0.75
DUP_JACCARD = 0.60


# ============================================================
# GPU / MEMORY HELPERS
# ============================================================

if torch.cuda.is_available():
    DEVICE = "cuda"
    USE_4BIT = True
    print("GPU:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    USE_4BIT = False
    print("WARNING: GPU not available.")


def clear_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass


def show_gpu_memory(label=""):
    if not torch.cuda.is_available():
        return
    try:
        allocated = torch.cuda.memory_allocated(0) / (1024 ** 3)
        reserved = torch.cuda.memory_reserved(0) / (1024 ** 3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"GPU Memory {label}: Allocated={allocated:.2f} GB | "
              f"Reserved={reserved:.2f} GB | Total={total:.2f} GB")
    except Exception:
        pass


clear_cuda_memory()
show_gpu_memory("at startup")


# ============================================================
# AGENT 2 REQUIREMENTS
# ============================================================

if "agent2_result" not in globals():
    raise NameError("agent2_result is not available. Run Agent 2 first.")

requirements = agent2_result.get("requirements", {})
if not requirements:
    raise ValueError("Agent 2 requirements are empty.")

print("\nAgent 2 Requirements:")
print(json.dumps(requirements, indent=4))

QUESTION_COUNT = requirements.get("Number")
QUESTION_TYPE = requirements.get("Type", requirements.get("Question Type"))
TOPIC = requirements.get("Topic")
DIFFICULTY = requirements.get("Difficulty")

if QUESTION_COUNT is None:
    raise ValueError("Agent 2 did not provide question count.")
QUESTION_COUNT = int(QUESTION_COUNT)


# ============================================================
# USER MODE
# ============================================================

print("\n" + "=" * 70)
print("QUESTION GENERATION MODE")
print("=" * 70)
print("1. text\n2. image\n3. mixed")

while True:
    generation_mode = input("\nQuestion type (text/image/mixed): ").strip().lower()
    if generation_mode in ["text", "image", "mixed"]:
        break
    print("Please enter text, image, or mixed.")

if generation_mode == "text":
    TEXT_COUNT, IMAGE_COUNT = QUESTION_COUNT, 0
elif generation_mode == "image":
    TEXT_COUNT, IMAGE_COUNT = 0, QUESTION_COUNT
else:
    while True:
        try:
            IMAGE_COUNT = int(input(
                f"\nHow many of {QUESTION_COUNT} questions should be image-based? "))
            if 0 <= IMAGE_COUNT <= QUESTION_COUNT:
                break
            print(f"Enter a value between 0 and {QUESTION_COUNT}.")
        except ValueError:
            print("Please enter a valid number.")
    TEXT_COUNT = QUESTION_COUNT - IMAGE_COUNT

print("\n" + "=" * 70)
print("GENERATION CONFIGURATION")
print("=" * 70)
print("Total questions :", QUESTION_COUNT)
print("Mode            :", generation_mode)
print("Text / Image    :", TEXT_COUNT, "/", IMAGE_COUNT)
print("Topic           :", TOPIC)
print("Difficulty      :", DIFFICULTY)
print("Question type   :", QUESTION_TYPE)
print("Self-check      :", VERIFY_TEXT)
print("=" * 70)


# ============================================================
# DIFFICULTY GUIDANCE (text questions)
# ============================================================

def difficulty_guidance(difficulty):
    d = str(difficulty).strip().lower()

    if d in ["easy", "simple", "basic", "beginner"]:
        return (
            "DIFFICULTY = EASY (JEE Main easy level)\n"
            "- direct application of one concept or one formula\n"
            "- one or two reasoning steps\n"
            "- distractors are clearly wrong to a well-prepared student"
        )

    if d in ["moderate", "medium", "intermediate", "average"]:
        return (
            "DIFFICULTY = MODERATE, i.e. MEDIUM-TO-HARD (upper half of a JEE Main paper)\n"
            "- every question needs at least TWO reasoning steps or the combination of "
            "TWO concepts\n"
            "- NO direct recall, NO definitions, NO 'which of the following is a ...' "
            "single-fact questions, NO naming of simple alkanes/alcohols\n"
            "- include exceptions, anomalous trends, multi-step reactions, "
            "calculations with several steps, or careful statement evaluation\n"
            "- distractors must be close and tempting (typical student mistakes)\n"
            "- explanations must show the actual reasoning or calculation"
        )

    if d in ["hard", "difficult", "tough", "advanced", "challenging"]:
        return (
            "DIFFICULTY = HARD (hardest JEE Main questions)\n"
            "- combine 2-3 concepts, multi-step derivations or reaction sequences\n"
            "- subtle exceptions and tricky distractors\n"
            "- no recall questions at all"
        )

    return (
        "DIFFICULTY = MIXED\n"
        "- roughly 30% easy, 40% medium, 30% hard\n"
        "- no purely trivial recall questions"
    )


DIFFICULTY_GUIDE = difficulty_guidance(DIFFICULTY)

# ============================================================
# MODERATE-TO-HARD JEE MAIN TARGET PROFILE
# ============================================================
# The old generator could call a question "Moderate" even when it only
# required one fact or one direct formula.  These controls make the
# requested difficulty measurable instead of relying only on wording.

MT_HARD_MIN_REASONING_STEPS = 2
MT_HARD_QUANTITATIVE_TARGET = 0.45
MT_HARD_MULTI_STEP_TARGET = 0.30
MT_HARD_APPLICATION_TARGET = 0.15
MT_HARD_MAX_DIRECT_SHARE = 0.20

QUANTITATIVE_TERMS = re.compile(
    r"\b(calculate|calculated|compute|determine|find|evaluate|estimate|moles?|mol|"
    r"molarity|molality|normality|concentration|pH|pOH|Ka|Kb|Ksp|Kc|Kp|rate|rate constant|"
    r"half-life|activation energy|enthalpy|entropy|equilibrium|pressure|volume|mass|density|"
    r"temperature|energy|wavelength|frequency|work function|potential|current|percentage|percent|"
    r"ratio|fraction|yield|osmotic pressure|vapour pressure|boiling point|freezing point)\b|"
    r"[=<>±×÷]", re.IGNORECASE)

MULTI_STEP_TERMS = re.compile(
    r"\b(then|after|before|subsequently|followed by|using the result|from this|hence|therefore|"
    r"first.*then|if.*then|mixture|successive|two-step|multi-step|combined|simultaneously|"
    r"in the next step|further|additional|both.*and)\b", re.IGNORECASE)

APPLICATION_TERMS = re.compile(
    r"\b(given|sample|mixture|solution|experimental|laboratory|industrial|process|data|graph|"
    r"table|observation|reaction sequence|unknown|compound X|compound Y|real|application)\b",
    re.IGNORECASE)

DIRECT_RECALL_PATTERNS = [
    re.compile(r"^\s*(which|what) (is|are) (the )?(definition|name|common name|formula)\b", re.I),
    re.compile(r"^\s*which of the following is (a|an)\b", re.I),
    re.compile(r"^\s*the (name|formula|hybridisation|hybridization|shape|geometry) of\b", re.I),
    re.compile(r"^\s*which (element|compound|species|ion|molecule)\b", re.I),
]


def assess_text_difficulty(q):
    """Heuristic quality gate for Moderate-to-Hard JEE Main questions."""
    question = str(q.get("question", ""))
    explanation = str(q.get("explanation", ""))
    blob = question + " " + explanation

    quantitative = bool(QUANTITATIVE_TERMS.search(question))
    multi_step = bool(MULTI_STEP_TERMS.search(blob))
    application = bool(APPLICATION_TERMS.search(question))
    direct = any(p.search(question) for p in DIRECT_RECALL_PATTERNS)

    # Long, calculation-heavy explanations are usually multi-step even when
    # the model does not explicitly write "then" or "next".
    equation_count = len(re.findall(r"=|≈|~|→|\btherefore\b|\bso\b", explanation, re.I))
    if equation_count >= 2 or len(explanation) >= 220:
        multi_step = True

    score = int(quantitative) + int(multi_step) + int(application) - int(direct and not quantitative)
    if quantitative and multi_step:
        category = "quantitative_multi_step"
    elif quantitative:
        category = "quantitative"
    elif multi_step or application:
        category = "application_reasoning"
    else:
        category = "conceptual_direct"

    return {
        "score": score,
        "quantitative": quantitative,
        "multi_step": multi_step,
        "application": application,
        "direct": direct,
        "category": category,
    }



# ============================================================
# JSON EXTRACTION
# ============================================================

def remove_thinking_blocks(text):
    if not text:
        return text
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()


def remove_markdown_fences(text):
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()


def extract_json(raw_response):
    if not raw_response:
        return None

    text = remove_markdown_fences(remove_thinking_blocks(str(raw_response)))
    first_brace = text.find("{")
    if first_brace == -1:
        return None

    try:
        data, _ = json.JSONDecoder().raw_decode(text[first_brace:])
        return data
    except Exception:
        pass

    last_brace = text.rfind("}")
    if last_brace != -1:
        try:
            return json.loads(text[first_brace:last_brace + 1])
        except Exception:
            pass

    return None


# ============================================================
# ################  IMAGE QUESTION ENGINE (no model)  ########
# ============================================================

# ---------- molecule banks ----------

ORGANIC_BANK = [
    ("aspirin", "CC(=O)Oc1ccccc1C(=O)O"),
    ("paracetamol", "CC(=O)Nc1ccc(O)cc1"),
    ("ibuprofen", "CC(C)Cc1ccc(cc1)C(C)C(=O)O"),
    ("caffeine", "Cn1cnc2c1c(=O)n(C)c(=O)n2C"),
    ("vanillin", "COc1cc(C=O)ccc1O"),
    ("cinnamaldehyde", "O=CC=Cc1ccccc1"),
    ("salicylaldehyde", "O=Cc1ccccc1O"),
    ("acetanilide", "CC(=O)Nc1ccccc1"),
    ("methyl benzoate", "COC(=O)c1ccccc1"),
    ("benzophenone", "O=C(c1ccccc1)c1ccccc1"),
    ("naphthalene", "c1ccc2ccccc2c1"),
    ("lactic acid", "CC(O)C(=O)O"),
    ("tartaric acid", "OC(C(O)C(=O)O)C(=O)O"),
    ("alanine", "CC(N)C(=O)O"),
    ("glucose (pyranose)", "OCC1OC(O)C(O)C(O)C1O"),
    ("menthol", "CC(C)C1CCC(C)CC1O"),
    ("camphor", "CC1(C)C2CCC1(C)C(=O)C2"),
    ("limonene", "CC1=CCC(CC1)C(C)=C"),
    ("citral", "CC(C)=CCCC(C)=CC=O"),
    ("but-2-ynoic acid", "CC#CC(=O)O"),
    ("phenylacetylene", "C#Cc1ccccc1"),
    ("styrene", "C=Cc1ccccc1"),
    ("cyclohexanone", "O=C1CCCCC1"),
    ("acetophenone", "CC(=O)c1ccccc1"),
    ("benzamide", "NC(=O)c1ccccc1"),
    ("nitrobenzene", "[O-][N+](=O)c1ccccc1"),
    ("p-nitrophenol", "Oc1ccc(cc1)[N+](=O)[O-]"),
    ("aniline", "Nc1ccccc1"),
    ("phthalic acid", "OC(=O)c1ccccc1C(=O)O"),
    ("maleic acid", "OC(=O)C=CC(=O)O"),
    ("crotonaldehyde", "CC=CC=O"),
    ("mesityl oxide", "CC(C)=CC(C)=O"),
    ("pyridine", "c1ccncc1"),
    ("furfural", "O=Cc1ccco1"),
    ("adrenaline", "CNCC(O)c1ccc(O)c(O)c1"),
    ("ephedrine", "CNC(C)C(O)c1ccccc1"),
    ("2-methylbutan-1-ol", "CCC(C)CO"),
    ("2,3-dichlorobutane", "CC(Cl)C(Cl)C"),
    ("bromochlorofluoromethane", "FC(Cl)Br"),
    ("buta-1,3-diene", "C=CC=C"),
    ("cyclopentadiene", "C1=CC=CC1"),
    ("anthracene", "c1ccc2cc3ccccc3cc2c1"),
    ("acrylonitrile", "C=CC#N"),
    ("benzoic acid", "OC(=O)c1ccccc1"),
    ("ethyl acetoacetate", "CCOC(=O)CC(C)=O"),
    ("p-toluidine", "Cc1ccc(N)cc1"),
    ("salicylic acid", "OC(=O)c1ccccc1O"),
    ("benzyl alcohol", "OCc1ccccc1"),
    ("methyl vinyl ketone", "C=CC(C)=O"),
    ("2-butyne-1,4-diol", "OCC#CCO"),
    ("cyclohexene", "C1=CCCCC1"),
]

INORGANIC_BANK = [
    ("SF4", "FS(F)(F)F"),
    ("XeF4", "F[Xe](F)(F)F"),
    ("XeF2", "F[Xe]F"),
    ("SF6", "FS(F)(F)(F)(F)F"),
    ("PCl5", "ClP(Cl)(Cl)(Cl)Cl"),
    ("PF5", "FP(F)(F)(F)F"),
    ("XeO3", "O=[Xe](=O)=O"),
    ("XeOF4", "O=[Xe](F)(F)(F)F"),
    ("XeO2F2", "O=[Xe](=O)(F)F"),
    ("IF5", "FI(F)(F)(F)F"),
    ("BF3", "FB(F)F"),
    ("NF3", "FN(F)F"),
    ("NH3", "N"),
    ("H2O", "O"),
    ("CH4", "C"),
    ("SO2", "O=S=O"),
    ("SO3", "O=S(=O)=O"),
    ("BeCl2", "Cl[Be]Cl"),
    ("CO2", "O=C=O"),
    ("SCl2", "ClSCl"),
    ("SiF4", "F[Si](F)(F)F"),
    ("IF7", "FI(F)(F)(F)(F)(F)F"),
]


# ---------- RDKit fact helpers ----------

FUNCTIONAL_GROUPS = {
    "carboxylic acid": "[CX3](=O)[OX2H1]",
    "ester": "[CX3](=O)[OX2][#6]",
    "amide": "[CX3](=O)[NX3]",
    "aldehyde": "[CX3H1](=O)[#6]",
    "ketone": "[#6][CX3](=O)[#6]",
    "alcohol": "[CX4][OX2H]",
    "phenol": "c[OX2H]",
    "ether": "[OX2;!$(O[CX3]=O)]([#6])[#6]",
    "amine": "[NX3;!$(NC=O);!$(N=*);!$([N+])]",
    "nitro": "[N+](=O)[O-]",
    "alkene": "[CX3]=[CX3]",
    "alkyne": "[CX2]#[CX2]",
    "nitrile": "C#N",
    "aromatic ring": "a1aaaaa1",
}

GROUP_PATTERNS = {k: Chem.MolFromSmarts(v) for k, v in FUNCTIONAL_GROUPS.items()}


def organic_facts(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("invalid smiles")

    molH = Chem.AddHs(mol)

    sp = sp2 = sp3 = 0
    for atom in mol.GetAtoms():
        if atom.GetSymbol() != "C":
            continue
        h = atom.GetHybridization()
        if h == Chem.HybridizationType.SP:
            sp += 1
        elif h == Chem.HybridizationType.SP2:
            sp2 += 1
        elif h == Chem.HybridizationType.SP3:
            sp3 += 1

    kek = Chem.Mol(molH)
    Chem.Kekulize(kek, clearAromaticFlags=True)
    sigma = pi = 0
    for b in kek.GetBonds():
        bt = b.GetBondType()
        if bt == Chem.BondType.SINGLE:
            sigma += 1
        elif bt == Chem.BondType.DOUBLE:
            sigma += 1
            pi += 1
        elif bt == Chem.BondType.TRIPLE:
            sigma += 1
            pi += 2
        else:
            raise ValueError("unexpected bond type")

    h_bonds = sum(1 for b in molH.GetBonds()
                  if b.GetBeginAtom().GetAtomicNum() == 1 or b.GetEndAtom().GetAtomicNum() == 1)

    chiral = len(Chem.FindMolChiralCenters(mol, includeUnassigned=True,
                                           useLegacyImplementation=False))

    groups = [name for name, patt in GROUP_PATTERNS.items() if mol.HasSubstructMatch(patt)]

    return {
        "sp": sp, "sp2": sp2, "sp3": sp3,
        "sigma": sigma, "pi": pi, "sigma_no_h": sigma - h_bonds,
        "chiral": chiral,
        "rings": mol.GetRingInfo().NumRings(),
        "formula": rdMolDescriptors.CalcMolFormula(mol),
        "groups": groups,
    }


HYB_NAMES = {2: "sp", 3: "sp\u00b2", 4: "sp\u00b3", 5: "sp\u00b3d",
             6: "sp\u00b3d\u00b2", 7: "sp\u00b3d\u00b3"}

SHAPES = {
    (2, 0): "Linear",
    (3, 0): "Trigonal planar",
    (3, 1): "Bent (V-shaped)",
    (4, 0): "Tetrahedral",
    (4, 1): "Trigonal pyramidal",
    (4, 2): "Bent (V-shaped)",
    (5, 0): "Trigonal bipyramidal",
    (5, 1): "See-saw",
    (5, 2): "T-shaped",
    (5, 3): "Linear",
    (6, 0): "Octahedral",
    (6, 1): "Square pyramidal",
    (6, 2): "Square planar",
    (7, 0): "Pentagonal bipyramidal",
}


def central_atom_info(smiles):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    heavy = [a for a in mol.GetAtoms() if a.GetAtomicNum() > 1]
    max_deg = max(a.GetDegree() for a in heavy)
    candidates = [a for a in heavy if a.GetDegree() == max_deg]
    if len(candidates) != 1:
        raise ValueError("no unique central atom")
    atom = candidates[0]

    bond_order_sum = int(sum(b.GetBondTypeAsDouble() for b in atom.GetBonds()))
    valence_e = Chem.GetPeriodicTable().GetNOuterElecs(atom.GetAtomicNum())
    remaining = valence_e - bond_order_sum - atom.GetFormalCharge()
    if remaining < 0 or remaining % 2 != 0:
        raise ValueError("odd electron count")

    lone_pairs = remaining // 2
    sigma = atom.GetDegree()
    steric = sigma + lone_pairs

    if steric not in HYB_NAMES or (steric, lone_pairs) not in SHAPES:
        raise ValueError("unsupported geometry")

    return {
        "symbol": atom.GetSymbol(), "valence_e": valence_e, "sigma": sigma,
        "bond_order_sum": bond_order_sum, "lone_pairs": lone_pairs,
        "steric": steric, "hybridisation": HYB_NAMES[steric],
        "shape": SHAPES[(steric, lone_pairs)],
    }


# ---------- option helpers ----------

def int_options(correct, extras=()):
    pool = []
    for c in list(extras) + [correct - 1, correct + 1, correct - 2, correct + 2,
                             correct + 3, correct - 3]:
        if c >= 0 and c != correct and c not in pool:
            pool.append(c)
    return [str(x) for x in pool[:3]]


def _is_number(text):
    try:
        float(str(text).split()[0])
        return True
    except Exception:
        return False


IMAGE_COUNTER = 0
USED_IMAGE_KEYS = set()
USED_MOLECULES = set()
FAMILY_USE = Counter()


def next_image_path(prefix):
    global IMAGE_COUNTER
    IMAGE_COUNTER += 1
    return os.path.join(IMAGE_OUTPUT_FOLDER, f"{prefix}_{IMAGE_COUNTER:03d}.png")


def assemble_image_question(question, correct, distractors, explanation, image_path,
                            image_type, image_description, smiles=None):
    options = [str(correct)] + [str(d) for d in distractors][:3]
    if len(set(options)) != 4:
        raise ValueError("options not distinct")

    if all(_is_number(o) for o in options):
        options.sort(key=lambda o: float(o.split()[0]))
    else:
        random.shuffle(options)

    q = {
        "question": question,
        "option_A": options[0], "option_B": options[1],
        "option_C": options[2], "option_D": options[3],
        "correct_answer": "ABCD"[options.index(str(correct))],
        "explanation": explanation,
        "question_type": "image",
        "image_required": True,
        "image_type": image_type,
        "image_description": image_description,
        "image_answerable": True,
        "generated_image": image_path,
        "image_status": "generated",
        "verification": "computed",
        "difficulty_category": "image_calculation" if image_type == "graph" else "image_reasoning",
        "reasoning_steps": 2,
    }
    if smiles:
        q["smiles"] = smiles
    return q


def draw_molecule(smiles, path):
    mol = Chem.MolFromSmiles(smiles)
    Draw.MolToFile(mol, path, size=(600, 450))


def pick_molecule(bank):
    unused = [m for m in bank if m[1] not in USED_MOLECULES]
    return random.choice(unused if unused else bank)


# ---------- organic families ----------

def build_organic(family):
    name, smiles = pick_molecule(ORGANIC_BANK)
    f = organic_facts(smiles)
    formula = f["formula"]
    intro = f"The structure shown is {name} ({formula})."

    if family == "org_sp2":
        if f["sp2"] < 2:
            raise ValueError("ineligible")
        question = random.choice([
            "How many carbon atoms in the structure shown are sp\u00b2 hybridised?",
            "The number of sp\u00b2 hybridised carbon atoms present in the structure shown is",
        ])
        correct = str(f["sp2"])
        distractors = int_options(f["sp2"], extras=[f["sp2"] + f["sp"]] if f["sp"] else [])
        expl = (f"{intro} Carbons in C=C, C=O and aromatic rings are sp\u00b2. "
                f"Counting gives {f['sp2']} sp\u00b2 carbons ({f['sp3']} sp\u00b3, {f['sp']} sp).")

    elif family == "org_sp3":
        if f["sp3"] < 2:
            raise ValueError("ineligible")
        question = random.choice([
            "How many carbon atoms in the structure shown are sp\u00b3 hybridised?",
            "The number of sp\u00b3 hybridised carbon atoms in the structure shown is",
        ])
        correct = str(f["sp3"])
        distractors = int_options(f["sp3"])
        expl = (f"{intro} Carbons with four single bonds are sp\u00b3. "
                f"There are {f['sp3']} sp\u00b3, {f['sp2']} sp\u00b2 and {f['sp']} sp carbons.")

    elif family == "org_chiral":
        if f["chiral"] < 1:
            raise ValueError("ineligible")
        question = random.choice([
            "The number of chiral centres (stereogenic carbons) in the structure shown is",
            "How many asymmetric carbon atoms are present in the structure shown?",
        ])
        correct = str(f["chiral"])
        distractors = int_options(f["chiral"])
        expl = (f"{intro} A chiral centre is a carbon attached to four different groups. "
                f"The structure has {f['chiral']} such centre(s).")

    elif family == "org_sigma":
        question = random.choice([
            "The total number of sigma (\u03c3) bonds in the structure shown, "
            "including all C\u2013H and other bonds to hydrogen, is",
            "How many \u03c3 bonds (counting every bond to hydrogen as well) are present "
            "in the structure shown?",
        ])
        correct = str(f["sigma"])
        distractors = int_options(f["sigma"], extras=[f["sigma_no_h"]])
        expl = (f"{intro} Every single bond is one \u03c3 bond, a double bond has one \u03c3 and one \u03c0, "
                f"a triple bond has one \u03c3 and two \u03c0. Counting all bonds including those to "
                f"hydrogen gives {f['sigma']} \u03c3 bonds ({f['sigma_no_h']} without bonds to H).")

    elif family == "org_pi":
        if f["pi"] < 2:
            raise ValueError("ineligible")
        question = random.choice([
            "The total number of pi (\u03c0) bonds in the structure shown is",
            "How many \u03c0 bonds are present in the structure shown?",
        ])
        correct = str(f["pi"])
        distractors = int_options(f["pi"])
        expl = (f"{intro} Each double bond has one \u03c0 bond and each triple bond has two; "
                f"a benzene ring contributes three. Total \u03c0 bonds = {f['pi']}.")

    elif family == "org_diff":
        diff = f["sigma"] - f["pi"]
        question = ("The difference between the number of \u03c3 bonds and the number of \u03c0 bonds "
                    "(counting all bonds, including those to hydrogen) in the structure shown is")
        correct = str(diff)
        distractors = int_options(diff, extras=[f["sigma"] + f["pi"]])
        expl = (f"{intro} \u03c3 bonds = {f['sigma']} and \u03c0 bonds = {f['pi']}, "
                f"so the difference is {f['sigma']} \u2212 {f['pi']} = {diff}.")

    elif family == "org_rings":
        if f["rings"] < 2:
            raise ValueError("ineligible")
        question = "The total number of rings present in the structure shown is"
        correct = str(f["rings"])
        distractors = int_options(f["rings"])
        expl = f"{intro} Counting the smallest set of smallest rings gives {f['rings']} rings."

    elif family == "org_formula":
        counts = Counter({el: int(n or 1) for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", formula)})

        def fmt(c):
            parts = []
            for el in ["C", "H"]:
                if c.get(el, 0) > 0:
                    parts.append(el + (str(c[el]) if c[el] > 1 else ""))
            for el in sorted(k for k in c if k not in ("C", "H") and c[k] > 0):
                parts.append(el + (str(c[el]) if c[el] > 1 else ""))
            return "".join(parts)

        variants = []
        for dh in (2, -2):
            c = Counter(counts); c["H"] += dh
            variants.append(c)
        for do in (1, -1):
            c = Counter(counts); c["O"] += do
            variants.append(c)
        c = Counter(counts); c["C"] += 1; c["H"] += 2
        variants.append(c)
        c = Counter(counts); c["C"] -= 1; c["H"] -= 2
        variants.append(c)

        cand = []
        for v in variants:
            if min(v.values()) >= 0 and v["C"] >= 1 and v["H"] >= 1:
                s = fmt(v)
                if s != formula and s not in cand:
                    cand.append(s)
        if len(cand) < 3:
            raise ValueError("not enough distractors")

        question = "The molecular formula of the compound shown is"
        correct = formula
        distractors = random.sample(cand, 3)
        expl = (f"{intro} Counting every atom (including implicit hydrogens) in the "
                f"structure gives {formula}.")

    elif family == "org_dbe":
        counts = Counter({el: int(n or 1) for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", formula)})
        C = counts.get("C", 0)
        H = counts.get("H", 0)
        N = counts.get("N", 0)
        X = sum(counts.get(el, 0) for el in ["F", "Cl", "Br", "I"])
        dbe = (2 * C + 2 + N - H - X) / 2
        if dbe < 1 or abs(dbe - round(dbe)) > 1e-9:
            raise ValueError("ineligible")
        dbe = int(round(dbe))
        rings = f["rings"]
        if dbe < rings:
            raise ValueError("ineligible")
        pi_bonds = dbe - rings
        question = (
            f"For the structure shown, the degree of unsaturation (DBE) is {dbe}. "
            f"If the {rings} ring(s) account for the ring contribution, how many \u03c0 bonds "
            "must be present?"
        )
        correct = str(pi_bonds)
        distractors = int_options(pi_bonds, extras=[dbe, rings, f["pi"]])
        expl = (f"For {formula}, DBE = (2C + 2 + N - H - X)/2 = {dbe}. "
                f"DBE equals rings + \u03c0 bonds, so \u03c0 bonds = {dbe} - {rings} = {pi_bonds}. "
                f"The structure also contains {f['pi']} \u03c0 bond(s), confirming the result.")

    elif family == "org_combustion":
        counts = Counter({el: int(n or 1) for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", formula)})
        if any(counts.get(el, 0) for el in ["F", "Cl", "Br", "I"]) or counts.get("C", 0) == 0:
            raise ValueError("ineligible")
        C = counts.get("C", 0)
        H = counts.get("H", 0)
        O = counts.get("O", 0)
        # Complete combustion to CO2, H2O and (where applicable) N2.
        o2_per_mol = C + H / 4 - O / 2
        if o2_per_mol <= 0:
            raise ValueError("ineligible")
        sample_mol = random.choice([0.20, 0.25, 0.40, 0.50])
        required_o2 = sample_mol * o2_per_mol
        question = (
            f"The compound in the structure shown has formula {formula}. If {sample_mol:g} mol "
            "of the compound undergoes complete combustion, the amount of O2 required is"
        )
        correct = f"{required_o2:.3g} mol"
        distractors = [
            f"{sample_mol * (C + H / 2 - O):.3g} mol",
            f"{sample_mol * (C + H / 4):.3g} mol",
            f"{sample_mol * (C + H / 4 + O / 2):.3g} mol",
        ]
        distractors = list(dict.fromkeys(distractors))
        if len(distractors) < 3 or str(correct) in distractors:
            raise ValueError("bad distractors")
        expl = (f"For CxHyOz, O2 required per mole = x + y/4 - z/2 = "
                f"{C} + {H}/4 - {O}/2 = {o2_per_mol:.3g} mol. "
                f"For {sample_mol:g} mol, O2 = {sample_mol:g} \u00d7 {o2_per_mol:.3g} = {required_o2:.3g} mol.")

    elif family == "org_molar_mass":
        counts = Counter({el: int(n or 1) for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", formula)})
        atomic_masses = {
            "H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999,
            "F": 18.998, "Cl": 35.45, "Br": 79.904, "I": 126.904
        }
        if any(el not in atomic_masses for el in counts):
            raise ValueError("unsupported element")
        molar_mass = sum(atomic_masses[el] * n for el, n in counts.items())
        sample_mol = random.choice([0.15, 0.20, 0.25, 0.40])
        sample_mass = sample_mol * molar_mass
        question = (
            f"The structure shown has molecular formula {formula}. What mass of the compound "
            f"is present in {sample_mol:g} mol of the compound?"
        )
        correct = f"{sample_mass:.3g} g"
        distractors = [
            f"{molar_mass:.3g} g",
            f"{sample_mass / 2:.3g} g",
            f"{sample_mass * 2:.3g} g",
        ]
        expl = (f"Molar mass of {formula} = {molar_mass:.3f} g mol^-1. "
                f"Using m = nM, m = {sample_mol:g} \u00d7 {molar_mass:.3f} = {sample_mass:.3g} g.")

    elif family == "org_groups":
        groups = f["groups"]
        if len(groups) < 2 or len(groups) > 4:
            raise ValueError("ineligible")
        absent = [g for g in FUNCTIONAL_GROUPS if g not in groups]
        correct = ", ".join(sorted(groups))
        distractors = []
        tries = 0
        while len(distractors) < 3 and tries < 60:
            tries += 1
            s = list(groups)
            s[random.randrange(len(s))] = random.choice(absent)
            d = ", ".join(sorted(set(s)))
            if len(set(s)) == len(groups) and d != correct and d not in distractors:
                distractors.append(d)
        if len(distractors) < 3:
            raise ValueError("not enough distractors")
        question = ("Which of the following correctly lists ALL the functional groups "
                    "present in the structure shown?")
        expl = f"{intro} It contains: {correct}."

    else:
        raise ValueError("unknown family")

    key = (smiles, family)
    if key in USED_IMAGE_KEYS:
        raise ValueError("already used")

    path = next_image_path("mol")
    draw_molecule(smiles, path)
    q = assemble_image_question(
        question, correct, distractors, expl, path, "molecular_structure",
        f"Structure of an organic molecule ({family})", smiles=smiles)

    USED_IMAGE_KEYS.add(key)
    USED_MOLECULES.add(smiles)
    return q


# ---------- inorganic families ----------

def build_inorganic(family):
    name, smiles = pick_molecule(INORGANIC_BANK)
    info = central_atom_info(smiles)

    key = (smiles, family)
    if key in USED_IMAGE_KEYS:
        raise ValueError("already used")

    sym = info["symbol"]
    reason = (f"The central atom {sym} has {info['valence_e']} valence electrons and forms "
              f"{info['sigma']} \u03c3 bonds (total bond order {info['bond_order_sum']}), leaving "
              f"{info['lone_pairs']} lone pair(s). Steric number = {info['sigma']} + "
              f"{info['lone_pairs']} = {info['steric']}.")

    if family == "inorg_hyb":
        question = random.choice([
            "What is the hybridisation of the central atom in the structure shown?",
            "The central atom of the species shown is",
        ])
        if question.startswith("The central atom"):
            question = "The hybridisation state of the central atom in the species shown is"
        correct = info["hybridisation"]
        others = sorted((n for n in HYB_NAMES if HYB_NAMES[n] != correct),
                        key=lambda n: (abs(n - info["steric"]), random.random()))
        distractors = [HYB_NAMES[n] for n in others[:3]]
        expl = f"{reason} Hence the hybridisation is {correct}."

    elif family == "inorg_shape":
        question = "The shape (molecular geometry) of the species shown is"
        correct = info["shape"]
        same_sn = [s for (sn, lp), s in SHAPES.items()
                   if sn == info["steric"] and s != correct]
        others = [s for s in set(SHAPES.values()) if s != correct and s not in same_sn]
        random.shuffle(others)
        distractors = []
        for s in same_sn + others:
            if s not in distractors:
                distractors.append(s)
            if len(distractors) == 3:
                break
        expl = (f"{reason} With {info['lone_pairs']} lone pair(s) the shape is {correct}.")

    elif family == "inorg_lp":
        question = "The number of lone pairs of electrons on the central atom of the species shown is"
        correct = str(info["lone_pairs"])
        distractors = int_options(info["lone_pairs"])
        expl = reason + f" Lone pairs = {info['lone_pairs']}."

    elif family == "inorg_combined":
        # A two-step structure question: infer steric number, then use it to
        # identify the geometry. This is harder than asking either fact alone.
        question = (
            "From the structure shown, determine the steric number of the central atom "
            "and hence identify its molecular geometry."
        )
        correct = f"SN = {info['steric']}; {info['shape']}"
        alternatives = []
        for sn, shape in sorted(SHAPES.items(), key=lambda x: (abs(x[0][0] - info['steric']), random.random())):
            if shape == info["shape"] and sn[0] == info["steric"]:
                continue
            item = f"SN = {sn[0]}; {shape}"
            if item not in alternatives:
                alternatives.append(item)
            if len(alternatives) == 3:
                break
        distractors = alternatives
        expl = (f"The central atom has {info['sigma']} sigma bonds and {info['lone_pairs']} lone pair(s). "
                f"Steric number = {info['sigma']} + {info['lone_pairs']} = {info['steric']}. "
                f"For this electron-domain arrangement, the molecular geometry is {info['shape']}.")

    else:
        raise ValueError("unknown family")

    path = next_image_path("inorg")
    draw_molecule(smiles, path)
    q = assemble_image_question(
        question, correct, distractors, expl, path, "molecular_structure",
        f"Structure of a main-group species ({family})", smiles=smiles)

    USED_IMAGE_KEYS.add(key)
    USED_MOLECULES.add(smiles)
    return q


# ---------- graph families (matplotlib) ----------

def _fmt(x, nd=2):
    s = f"{x:.{nd}f}"
    return s


def _finish_graph(fig, path):
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)


def _label_point(ax, x, y, text, dx=8, dy=8):
    ax.plot([x], [y], "o", color="crimson", zorder=5)
    ax.annotate(text, (x, y), textcoords="offset points", xytext=(dx, dy), fontsize=10)


def build_graph(family):
    if family == "g_first_order":
        t_half = random.choice([10, 15, 20, 25, 30, 40, 50, 60])
        A0 = random.choice([0.8, 1.0, 1.2, 1.6, 2.0])
        key = (family, t_half, A0)
        if key in USED_IMAGE_KEYS:
            raise ValueError("already used")

        k = math.log(2) / t_half
        t2 = 2 * t_half
        y1 = math.log(A0)
        y2 = y1 - k * t2
        y1r, y2r = round(y1, 2), round(y2, 2)

        fig, ax = plt.subplots(figsize=(6, 4))
        ts = [0, 1.5 * t2]
        ax.plot(ts, [y1 - k * t for t in ts], color="navy")
        _label_point(ax, 0, y1, f"(0, {_fmt(y1r)})", dx=10, dy=8)
        _label_point(ax, t2, y2, f"({t2}, {_fmt(y2r)})", dx=10, dy=8)
        ax.set_xlabel("Time, t (min)")
        ax.set_ylabel("ln [A]")
        ax.grid(alpha=0.3)

        k_est = (y1r - y2r) / t2
        th_est = 0.693 / k_est

        def mins(x):
            return f"{round(x, 1):g} min"

        question = ("The graph shows ln[A] versus time for a first-order reaction "
                    "A \u2192 products. The half-life of the reaction, estimated from the two "
                    "marked points, is closest to")
        correct = mins(t_half)
        distractors = [mins(2 * t_half), mins(t_half / 2), mins(1 / k)]
        expl = (f"Slope = (y\u2082 \u2212 y\u2081)/(t\u2082 \u2212 t\u2081) = ({_fmt(y2r)} \u2212 ({_fmt(y1r)}))/{t2} = "
                f"\u2212{k_est:.4f} min\u207b\u00b9, so k \u2248 {k_est:.4f} min\u207b\u00b9. "
                f"t\u00bd = 0.693/k \u2248 {th_est:.1f} min \u2248 {t_half} min.")
        path = next_image_path("graph")
        _finish_graph(fig, path)
        q = assemble_image_question(question, correct, distractors, expl, path, "graph",
                                    "ln[A] versus time plot for a first-order reaction")

    elif family == "g_arrhenius":
        Ea = random.choice([40, 50, 60, 75, 80, 90, 100])
        key = (family, Ea)
        if key in USED_IMAGE_KEYS:
            raise ValueError("already used")

        R = 8.314
        slope = -Ea * 1000 / R
        y_ref = random.uniform(0.5, 3.0)
        lnA = y_ref - slope * 3.3e-3
        x1, x2 = 3.00e-3, 3.40e-3
        y1, y2 = lnA + slope * x1, lnA + slope * x2
        y1r, y2r = round(y1, 2), round(y2, 2)

        fig, ax = plt.subplots(figsize=(6, 4))
        xs = [2.8e-3, 3.6e-3]
        ax.plot([x * 1e3 for x in xs], [lnA + slope * x for x in xs], color="navy")
        _label_point(ax, x1 * 1e3, y1, f"({_fmt(x1 * 1e3)}, {_fmt(y1r)})", dx=8, dy=8)
        _label_point(ax, x2 * 1e3, y2, f"({_fmt(x2 * 1e3)}, {_fmt(y2r)})", dx=8, dy=8)
        ax.set_xlabel("1/T  (\u00d7 10\u207b\u00b3 K\u207b\u00b9)")
        ax.set_ylabel("ln k")
        ax.grid(alpha=0.3)

        Ea_est = (y1r - y2r) / (x2 - x1) * R / 1000
        question = ("The graph shows ln k versus 1/T for a reaction. Using the two marked points "
                    "and R = 8.314 J mol\u207b\u00b9 K\u207b\u00b9, the activation energy of the reaction is "
                    "closest to")
        correct = f"{Ea} kJ/mol"
        distractors = [f"{round(Ea * 2.303)} kJ/mol", f"{round(Ea / 2.303)} kJ/mol",
                       f"{round(Ea * 2)} kJ/mol"]
        expl = (f"ln k = ln A \u2212 E\u2090/(RT), so slope = \u2212E\u2090/R. Slope = ({_fmt(y2r)} \u2212 ({_fmt(y1r)}))"
                f"/({_fmt(x2 * 1e3)} \u2212 {_fmt(x1 * 1e3)})\u00d710\u207b\u00b3 \u2248 {(y2r - y1r) / (x2 - x1):.0f} K. "
                f"E\u2090 = 8.314 \u00d7 {abs((y2r - y1r) / (x2 - x1)):.0f} J/mol \u2248 {Ea_est:.1f} kJ/mol \u2248 {Ea} kJ/mol.")
        path = next_image_path("graph")
        _finish_graph(fig, path)
        q = assemble_image_question(question, correct, distractors, expl, path, "graph",
                                    "Arrhenius plot (ln k versus 1/T)")

    elif family == "g_zero_order":
        A0 = random.choice([0.8, 1.0, 1.2, 1.6, 2.0])
        k = random.choice([0.005, 0.01, 0.02, 0.025, 0.04])
        t_end = A0 / k
        t1_options = [t for t in range(10, 400, 10) if 0.2 * t_end <= t <= 0.6 * t_end]
        if not t1_options:
            raise ValueError("no valid t1")
        t1 = random.choice(t1_options)
        key = (family, A0, k, t1)
        if key in USED_IMAGE_KEYS:
            raise ValueError("already used")

        y1 = A0 - k * t1
        fig, ax = plt.subplots(figsize=(6, 4))
        ts = [0, 1.6 * t1]
        ax.plot(ts, [A0 - k * t for t in ts], color="navy")
        _label_point(ax, 0, A0, f"(0, {_fmt(A0)})", dx=10, dy=-14)
        _label_point(ax, t1, y1, f"({t1}, {_fmt(y1, 3)})", dx=10, dy=8)
        ax.set_xlabel("Time, t (min)")
        ax.set_ylabel("[A]  (mol L\u207b\u00b9)")
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.3)

        t75 = 0.75 * A0 / k
        question = ("The graph shows the variation of [A] with time for a zero-order reaction. "
                    "Using the two marked points, the time required for 75% of A to react is")

        def mins(x):
            return f"{round(x, 1):g} min"

        correct = mins(t75)
        distractors = [mins(0.5 * A0 / k), mins(0.9 * A0 / k), mins(1.0 * A0 / k)]
        expl = (f"Slope = ({_fmt(y1, 3)} \u2212 {_fmt(A0)})/{t1} = \u2212{k:g} mol L\u207b\u00b9 min\u207b\u00b9, so k = {k:g}. "
                f"For zero order, [A]\u2080 \u2212 [A] = kt. For 75% reaction [A]\u2080 \u2212 [A] = 0.75 \u00d7 {A0:g}, "
                f"so t = 0.75 \u00d7 {A0:g}/{k:g} = {t75:.1f} min.")
        path = next_image_path("graph")
        _finish_graph(fig, path)
        q = assemble_image_question(question, correct, distractors, expl, path, "graph",
                                    "[A] versus time plot for a zero-order reaction")

    elif family == "g_exp_decay":
        t_half = random.choice([10, 15, 20, 25, 30, 40, 50])
        A0 = random.choice([0.64, 0.8, 1.6, 2.0])
        key = (family, t_half, A0)
        if key in USED_IMAGE_KEYS:
            raise ValueError("already used")

        k = math.log(2) / t_half
        T = 2 * t_half
        A_T = A0 / 4

        fig, ax = plt.subplots(figsize=(6, 4))
        ts = [i * 3.2 * t_half / 200 for i in range(201)]
        ax.plot(ts, [A0 * math.exp(-k * t) for t in ts], color="navy")
        _label_point(ax, 0, A0, f"(0, {A0:g})", dx=10, dy=-4)
        _label_point(ax, T, A_T, f"({T}, {A_T:g})", dx=10, dy=10)
        ax.set_xlabel("Time, t (min)")
        ax.set_ylabel("[A]  (mol L\u207b\u00b9)")
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.3)

        question = ("The graph shows the decay of the concentration of A in a first-order "
                    "reaction. Using the two marked points, the rate constant of the reaction "
                    "is closest to")
        correct = f"{k:.4f} min\u207b\u00b9"
        distractors = [f"{k / 2:.4f} min\u207b\u00b9", f"{2 * k:.4f} min\u207b\u00b9",
                       f"{1 / T:.4f} min\u207b\u00b9"]
        expl = (f"[A] falls from {A0:g} to {A_T:g} (a factor of 4) in {T} min, so "
                f"k = (1/t) ln([A]\u2080/[A]) = ln 4/{T} = {k:.4f} min\u207b\u00b9 "
                f"(equivalent to t\u00bd = {t_half} min).")
        path = next_image_path("graph")
        _finish_graph(fig, path)
        q = assemble_image_question(question, correct, distractors, expl, path, "graph",
                                    "Exponential decay of [A] for a first-order reaction")

    elif family == "g_first_order_two_step":
        t_half = random.choice([10, 15, 20, 25, 30, 40])
        A0 = random.choice([0.8, 1.0, 1.2, 1.6])
        k = math.log(2) / t_half
        t1 = t_half
        t2 = 2 * t_half
        y1 = math.log(A0 / 2)
        y2 = math.log(A0 / 4)
        key = (family, t_half, A0)
        if key in USED_IMAGE_KEYS:
            raise ValueError("already used")

        fig, ax = plt.subplots(figsize=(6, 4))
        xs = [0, 3 * t_half]
        ax.plot(xs, [math.log(A0) - k * x for x in xs], color="navy")
        _label_point(ax, 0, math.log(A0), f"(0, {_fmt(math.log(A0))})", dx=8, dy=8)
        _label_point(ax, t1, y1, f"({t1}, {_fmt(y1)})", dx=8, dy=8)
        _label_point(ax, t2, y2, f"({t2}, {_fmt(y2)})", dx=8, dy=8)
        ax.set_xlabel("Time, t (min)")
        ax.set_ylabel("ln [A]")
        ax.grid(alpha=0.3)

        future_t = 3 * t_half
        remaining = A0 * math.exp(-k * future_t)
        percent_reacted = (1 - remaining / A0) * 100
        question = (
            "The graph represents a first-order reaction. Using the marked half-life and the "
            "first-order rate law, what percentage of A has reacted after the time corresponding "
            "to the end of the plotted interval?"
        )
        correct = f"{percent_reacted:.0f}%"
        distractors = [f"{50:.0f}%", f"{75:.0f}%", f"{87.5:.0f}%"]
        expl = (f"From the graph, one half-life is {t_half} min. The plotted interval is {future_t} min = "
                f"3 half-lives. Therefore [A]/[A]0 = (1/2)^3 = 1/8, so fraction reacted = 7/8. "
                f"Percentage reacted = {percent_reacted:.0f}%.")
        path = next_image_path("graph")
        _finish_graph(fig, path)
        q = assemble_image_question(question, correct, distractors, expl, path, "graph",
                                    "First-order ln[A] versus time plot requiring a multi-step interpretation")

    elif family == "g_zero_order_two_step":
        A0 = random.choice([1.0, 1.2, 1.6, 2.0])
        k = random.choice([0.01, 0.02, 0.025, 0.04])
        t1 = random.choice([10, 20, 30, 40])
        if A0 - k * t1 <= 0:
            raise ValueError("ineligible")
        key = (family, A0, k, t1)
        if key in USED_IMAGE_KEYS:
            raise ValueError("already used")

        y1 = A0 - k * t1
        fig, ax = plt.subplots(figsize=(6, 4))
        t_end = A0 / k
        xs = [0, t_end]
        ax.plot(xs, [A0, 0], color="navy")
        _label_point(ax, 0, A0, f"(0, {_fmt(A0)})", dx=8, dy=-12)
        _label_point(ax, t1, y1, f"({t1}, {_fmt(y1, 2)})", dx=8, dy=8)
        ax.set_xlabel("Time, t (min)")
        ax.set_ylabel("[A] (mol L^-1)")
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.3)

        target_fraction = random.choice([0.25, 0.50, 0.75])
        target_t = (target_fraction * A0) / k
        question = (
            "The graph is for a zero-order reaction. First obtain k from the marked points and "
            f"then determine the time required for {target_fraction * 100:.0f}% of A to be consumed."
        )
        correct = f"{target_t:.1f} min"
        distractors = [f"{target_t / 2:.1f} min", f"{target_t * 0.75:.1f} min", f"{target_t * 1.25:.1f} min"]
        expl = (f"From the graph, k = ([A]0 - [A]t1)/t1 = ({A0:g} - {y1:.3g})/{t1} = {k:g} mol L^-1 min^-1. "
                f"For {target_fraction * 100:.0f}% consumption, kt = {target_fraction:.2g}[A]0, so "
                f"t = {target_fraction:.2g}({A0:g})/{k:g} = {target_t:.1f} min.")
        path = next_image_path("graph")
        _finish_graph(fig, path)
        q = assemble_image_question(question, correct, distractors, expl, path, "graph",
                                    "Zero-order concentration-time plot requiring slope extraction and a second calculation")

    else:
        raise ValueError("unknown family")

    USED_IMAGE_KEYS.add(key)
    return q


# ---------- family selection ----------

# Direct-count families are retained for variety, but the new calculation
# families are deliberately included so image questions are not mostly recall.
ORGANIC_FAMS = [
    "org_sp2", "org_sp3", "org_chiral", "org_sigma", "org_pi",
    "org_diff", "org_rings", "org_formula", "org_groups",
    "org_dbe", "org_combustion", "org_molar_mass"
]
INORGANIC_FAMS = ["inorg_hyb", "inorg_shape", "inorg_lp", "inorg_combined"]
GRAPH_FAMS = [
    "g_first_order", "g_arrhenius", "g_zero_order", "g_exp_decay",
    "g_first_order_two_step", "g_zero_order_two_step"
]


def family_pool(topic):
    t = str(topic).lower()
    if any(k in t for k in ["kinetic", "rate of reaction", "arrhenius"]):
        return GRAPH_FAMS
    if any(k in t for k in ["organic", "hydrocarbon", "isomer", "biomolecule", "amine",
                            "carbonyl", "aldehyde", "alcohol", "haloalkane", "aromatic",
                            "polymer", "iupac"]):
        return ORGANIC_FAMS
    if any(k in t for k in ["bonding", "p-block", "p block", "vsepr", "inorganic",
                            "hybrid", "molecular structure"]):
        return INORGANIC_FAMS
    # For ALL, deliberately oversample calculation/multi-step image families.
    # Direct-count families remain available for variety but no longer dominate.
    return (
        ["org_dbe", "org_combustion", "org_molar_mass"] * 3
        + ["inorg_combined"] * 2
        + ["g_first_order_two_step", "g_zero_order_two_step"] * 2
        + ORGANIC_FAMS + INORGANIC_FAMS + GRAPH_FAMS
    )


def build_image_question(pool, max_tries=300):
    """Always returns an image question. Tries other molecules/graphs on failure."""
    last_error = None
    for attempt in range(max_tries):
        least = min(FAMILY_USE[f] for f in pool)
        candidates = [f for f in pool if FAMILY_USE[f] == least]
        # after many failures, allow any family
        family = random.choice(candidates if attempt < 100 else pool)
        try:
            if family.startswith("org_"):
                q = build_organic(family)
            elif family.startswith("inorg_"):
                q = build_inorganic(family)
            else:
                q = build_graph(family)
            FAMILY_USE[family] += 1
            q["image_family"] = family
            return q
        except Exception as e:
            last_error = e
            continue

    # bank exhausted: allow reuse of molecules/graphs (different random facts) as last resort
    USED_IMAGE_KEYS.clear()
    USED_MOLECULES.clear()
    return build_image_question(pool, max_tries=max_tries) if last_error else None


# ============================================================
# ============================================================
# TEXT QUESTIONS: KNOWLEDGE GRAPH -> QWEN3-8B -> NEW QUESTION
# ============================================================
# IMPORTANT ARCHITECTURE
#   TEXT  : Knowledge Graph -> retrieve source question -> Qwen3-8B -> new question
#   IMAGE : existing RDKit / matplotlib generation code above (unchanged)
#
# The Knowledge Graph is the source of the chemistry question/concept.
# Qwen is used to transform that source into a NEW exam-ready question.
# KG image questions are excluded from the TEXT path so the generated text
# question never depends on a missing/unclear image.
# ============================================================

IMAGE_POOL = family_pool(TOPIC)

KG_PATH = "/kaggle/working/chemistry_knowledge_graph.pkl"
KG_TEXT_POOL_TARGET = max(50, TEXT_COUNT * 8) if TEXT_COUNT > 0 else 0


def _kg_norm(value):
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def _kg_node_type(attrs):
    return _kg_norm(attrs.get("type", attrs.get("node_type", attrs.get("kind", "")))).lower()


def _kg_get_question(attrs):
    for key in ["question", "text", "question_text"]:
        if attrs.get(key):
            return _kg_norm(attrs[key])
    return ""


def _kg_get_difficulty(G, node):
    attrs = G.nodes[node]
    for key in ["difficulty", "level"]:
        if attrs.get(key):
            return _kg_norm(attrs[key])
    try:
        for nb in G.neighbors(node):
            na = G.nodes[nb]
            typ = _kg_node_type(na)
            if typ == "difficulty" or "difficulty" in typ:
                val = na.get("value", na.get("name", na.get("difficulty", nb)))
                return _kg_norm(val)
    except Exception:
        pass
    return ""


def _kg_get_topic(G, node):
    attrs = G.nodes[node]
    for key in ["topic", "chapter"]:
        if attrs.get(key):
            return _kg_norm(attrs[key])
    try:
        for nb in G.neighbors(node):
            na = G.nodes[nb]
            typ = _kg_node_type(na)
            if typ == "topic" or "topic" in typ:
                return _kg_norm(na.get("name", na.get("value", nb)))
    except Exception:
        pass
    return ""


def _kg_get_subtopic(G, node):
    attrs = G.nodes[node]
    for key in ["subtopic", "concept"]:
        if attrs.get(key):
            return _kg_norm(attrs[key])
    try:
        for nb in G.neighbors(node):
            na = G.nodes[nb]
            typ = _kg_node_type(na)
            if typ == "subtopic" or "subtopic" in typ or typ == "concept":
                return _kg_norm(na.get("name", na.get("value", nb)))
    except Exception:
        pass
    return ""


def _kg_get_question_type(G, node):
    attrs = G.nodes[node]
    for key in ["question_type", "type_name"]:
        if attrs.get(key):
            return _kg_norm(attrs[key])
    try:
        for nb in G.neighbors(node):
            na = G.nodes[nb]
            typ = _kg_node_type(na)
            if "questiontype" in typ or typ == "question_type":
                return _kg_norm(na.get("name", na.get("value", nb)))
    except Exception:
        pass
    return ""


def _kg_get_options(G, node):
    attrs = G.nodes[node]
    direct = []
    for key in ["options", "choices"]:
        value = attrs.get(key)
        if isinstance(value, (list, tuple)):
            direct = [_kg_norm(x) for x in value if _kg_norm(x)]
            break
    if len(direct) >= 4:
        return direct[:4]

    options = []
    try:
        for nb in G.neighbors(node):
            na = G.nodes[nb]
            typ = _kg_node_type(na)
            if typ == "option" or "option" in typ:
                txt = _kg_norm(na.get("text", na.get("option", na.get("value", ""))))
                if txt:
                    options.append(txt)
    except Exception:
        pass
    return options[:4]


def _kg_get_correct_answer(G, node, options):
    attrs = G.nodes[node]
    for key in ["correct_answer", "correct_option", "answer", "answer_key"]:
        value = _kg_norm(attrs.get(key))
        if value:
            m = re.search(r"\b([ABCD])\b", value.upper())
            if m:
                return m.group(1)
            # Some KGs store the option text instead of the letter.
            if value in options:
                return "ABCD"[options.index(value)]

    try:
        for nb in G.neighbors(node):
            ed = G.get_edge_data(node, nb) or {}
            na = G.nodes[nb]
            rel = _kg_norm(ed.get("relation", ed.get("type", ""))).upper()
            if rel == "HAS_CORRECT_OPTION" or "CORRECT_OPTION" in rel:
                txt = _kg_norm(na.get("text", na.get("option", na.get("value", ""))))
                if txt in options:
                    return "ABCD"[options.index(txt)]
    except Exception:
        pass
    return ""


def _kg_has_image(G, node):
    attrs = G.nodes[node]
    for key in ["images", "image", "image_path", "image_paths"]:
        value = attrs.get(key)
        if value:
            if isinstance(value, (list, tuple, set)):
                if len(value) > 0:
                    return True
            else:
                return True
    try:
        for nb in G.neighbors(node):
            na = G.nodes[nb]
            typ = _kg_node_type(na)
            if "questionimage" in typ or typ == "image" or "image" in typ:
                return True
    except Exception:
        pass
    return False


def _topic_matches(value, requested):
    requested = _kg_norm(requested).lower()
    value = _kg_norm(value).lower()
    if not requested or requested == "all":
        return True
    if not value:
        return False
    return requested in value or value in requested


def _difficulty_matches(value, requested):
    requested = _kg_norm(requested).lower()
    value = _kg_norm(value).lower()
    if not requested or requested in {"all", "mixed"}:
        return True
    aliases = {
        "easy": {"easy", "simple", "basic", "beginner"},
        "moderate": {"moderate", "medium", "intermediate", "average"},
        "medium": {"moderate", "medium", "intermediate", "average"},
        "hard": {"hard", "difficult", "tough", "advanced", "challenging"},
    }
    wanted = aliases.get(requested, {requested})
    return any(x in value for x in wanted)


def load_text_kg_sources(required_count):
    """Load text-only source questions from the Chemistry KG."""
    if required_count <= 0:
        return []
    if not os.path.exists(KG_PATH):
        raise FileNotFoundError(
            f"Knowledge Graph not found: {KG_PATH}. Run Agent 2 first and create the KG.")

    with open(KG_PATH, "rb") as f:
        G = pickle.load(f)

    if not hasattr(G, "nodes"):
        raise TypeError("The Knowledge Graph pickle does not contain a NetworkX graph.")

    candidates = []
    seen_questions = set()

    for node, attrs in G.nodes(data=True):
        qtype = _kg_node_type(attrs)
        question = _kg_get_question(attrs)

        # Agent 2 v2 uses Question nodes with a 'question' attribute.
        if qtype and qtype not in {"question", "questions"} and "question" not in qtype:
            continue
        if not question:
            continue

        # Do not use KG image questions for the text path.
        if _kg_has_image(G, node):
            continue

        options = _kg_get_options(G, node)
        if len(options) != 4:
            continue

        correct = _kg_get_correct_answer(G, node, options)
        if correct not in "ABCD":
            continue

        topic = _kg_get_topic(G, node)
        subtopic = _kg_get_subtopic(G, node)
        difficulty = _kg_get_difficulty(G, node)
        question_type = _kg_get_question_type(G, node)

        if TOPIC and str(TOPIC).upper() != "ALL" and not _topic_matches(topic, TOPIC):
            continue

        key = re.sub(r"\s+", " ", question.lower()).strip()
        if key in seen_questions:
            continue
        seen_questions.add(key)

        candidates.append({
            "kg_question_id": str(node),
            "question": question,
            "options": options,
            "correct_answer": correct,
            "topic": topic,
            "subtopic": subtopic,
            "difficulty": difficulty,
            "question_type": question_type,
        })

    if not candidates:
        raise RuntimeError(
            "No suitable text-only questions were found in the Knowledge Graph for "
            f"Topic={TOPIC!r}, Difficulty={DIFFICULTY!r}. "
            "Make sure Agent 2 built the KG and that it contains text-only MCQs.")

    # Prefer the requested difficulty first. If the KG has too few exact matches,
    # use other text-only questions from the same topic rather than failing.
    exact = [x for x in candidates if _difficulty_matches(x["difficulty"], DIFFICULTY)]
    fallback = [x for x in candidates if x not in exact]
    random.shuffle(exact)
    random.shuffle(fallback)

    pool = exact + fallback
    return pool[:max(required_count, KG_TEXT_POOL_TARGET)]


KG_TEXT_SOURCES = load_text_kg_sources(TEXT_COUNT) if TEXT_COUNT > 0 else []

if TEXT_COUNT > 0:
    print("\n" + "=" * 70)
    print("TEXT SOURCE: KNOWLEDGE GRAPH")
    print("=" * 70)
    print("KG path              :", KG_PATH)
    print("Text-only KG sources :", len(KG_TEXT_SOURCES))
    print("Topic filter         :", TOPIC)
    print("Difficulty preference:", DIFFICULTY)
    print("LLM                   :", MODEL_NAME)
    print("=" * 70)

    # ---------- Qwen3-8B ----------
    print("\nLoading tokenizer for KG -> LLM generation...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    print("Tokenizer loaded.")

    if globals().get("model") is not None:
        print("\nModel already loaded in this session - reusing it.")
    else:
        print("\nLoading Qwen3-8B...")
        clear_cuda_memory()

        if USE_4BIT:
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True
            )
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                quantization_config=quantization_config,
                device_map="auto",
                trust_remote_code=True
            )
        else:
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                torch_dtype=torch.float32,
                device_map="auto",
                trust_remote_code=True
            )
        model.eval()
        print("Model loaded successfully.")

    try:
        INPUT_DEVICE = next(model.parameters()).device
    except Exception:
        INPUT_DEVICE = torch.device(DEVICE)

    print("Model input device:", INPUT_DEVICE)
    show_gpu_memory("after model loading")
else:
    print("\nImage-only mode: no model needed (questions are built by code).")
    tokenizer = None
    model = None
    INPUT_DEVICE = torch.device(DEVICE)


SYSTEM_MESSAGE = (
    "You are an expert JEE Main Chemistry question-setter. "
    "The provided Knowledge Graph question is the factual source. "
    "Create a NEW question from that source while preserving the underlying "
    "chemistry concept and scientific correctness. The target is Moderate-to-Hard "
    "JEE Main, not JEE Advanced. Prefer numerical, equation-based, multi-step, "
    "data/graph interpretation and application questions over direct recall. Return JSON only."
)


def run_generation(prompt, tag, temperature, max_new_tokens=None, system=SYSTEM_MESSAGE):
    clear_cuda_memory()

    messages = [{"role": "system", "content": system},
                {"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True, enable_thinking=False
    ).to(INPUT_DEVICE)

    n_in = inputs["input_ids"].shape[-1]
    start = time.time()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens or MAX_NEW_TOKENS,
            do_sample=True,
            temperature=temperature,
            top_p=TOP_P,
            top_k=TOP_K,
            repetition_penalty=REPETITION_PENALTY,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0, n_in:]
    raw = tokenizer.decode(generated, skip_special_tokens=True)

    print(f"Input tokens: {n_in} | Generated: {len(generated)} | "
          f"Time: {time.time() - start:.1f}s")

    if tag:
        with open(f"/kaggle/working/agent4_{tag}_raw.txt", "w", encoding="utf-8") as f:
            f.write(raw)

    del outputs, generated, inputs
    clear_cuda_memory()
    return raw


# ---------- independent self-check ----------
def verify_questions(questions):
    """Model solves each generated question without seeing the answer key."""
    if not questions:
        return []

    prompts = []
    for q in questions:
        user = (
            "Solve this JEE Main Chemistry MCQ yourself. Work carefully, using at most "
            "200 words, then finish with exactly this line:\n"
            "FINAL ANSWER: <A, B, C or D>\n\n"
            f"Question: {q['question']}\n"
            f"A) {q['option_A']}\nB) {q['option_B']}\n"
            f"C) {q['option_C']}\nD) {q['option_D']}"
        )
        text = tokenizer.apply_chat_template(
            [{"role": "system", "content": "You are an expert chemistry teacher."},
             {"role": "user", "content": user}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False)
        prompts.append(text)

    verdicts = [None] * len(prompts)
    try:
        clear_cuda_memory()
        tokenizer.padding_side = "left"
        enc = tokenizer(prompts, return_tensors="pt", padding=True,
                        add_special_tokens=False).to(INPUT_DEVICE)
        n_in = enc["input_ids"].shape[-1]

        with torch.inference_mode():
            out = model.generate(
                **enc,
                max_new_tokens=VERIFY_MAX_TOKENS,
                do_sample=True,
                temperature=0.3,
                top_p=0.9,
                top_k=20,
                pad_token_id=tokenizer.pad_token_id
            )

        for i in range(len(prompts)):
            text = remove_thinking_blocks(
                tokenizer.decode(out[i, n_in:], skip_special_tokens=True))
            found = re.findall(r"FINAL ANSWER:\s*\**\s*\(?([ABCD])\b", text, re.IGNORECASE)
            verdicts[i] = found[-1].upper() if found else None

        del out, enc
        clear_cuda_memory()
    except Exception as e:
        print("Self-check failed:", type(e).__name__, e)
        clear_cuda_memory()

    return verdicts


# ---------- structural checks ----------
REQUIRED_FIELDS = ["question", "option_A", "option_B", "option_C",
                   "option_D", "correct_answer", "explanation"]

IMAGE_FIELDS = ["image_required", "image_type", "image_description",
                "image_answerable", "smiles"]

TEXT_REFERS_TO_IMAGE = re.compile(
    r"\b(figure|diagram|image|picture)\b|structure (shown|given below)|shown below|"
    r"graph shown|following (structure|graph)", re.IGNORECASE)


def norm_words(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))


def question_blob(q):
    return " ".join(str(q.get(k, "")) for k in
                    ["question", "option_A", "option_B", "option_C", "option_D"]).lower()


def is_duplicate(q, existing):
    blob = question_blob(q)
    words = norm_words(blob)
    stem = str(q["question"]).lower().strip()
    for e in existing:
        if difflib.SequenceMatcher(None, stem, str(e["question"]).lower().strip()).ratio() > 0.88:
            return True
        e_blob = question_blob(e)
        if difflib.SequenceMatcher(None, blob, e_blob).ratio() > DUP_SEQ_RATIO:
            return True
        e_words = norm_words(e_blob)
        if words and e_words and len(words & e_words) / len(words | e_words) > DUP_JACCARD:
            return True
    return False


def shuffle_options(q):
    letters = "ABCD"
    texts = [q[f"option_{l}"] for l in letters]
    blob = " ".join(texts) + " " + q["explanation"]
    if re.search(r"\b(both|all of the above|none of|neither|option\s*[A-D])\b|"
                 r"\([A-D]\)|\b[A-D]\s*(and|&)\s*[A-D]\b", blob, re.IGNORECASE):
        return
    if sum(1 for t in texts if re.match(r"^\s*[-+]?\d", t)) >= 3:
        return
    correct_index = letters.index(q["correct_answer"])
    order = list(range(4))
    random.shuffle(order)
    for new_pos, old_pos in enumerate(order):
        q[f"option_{letters[new_pos]}"] = texts[old_pos]
    q["correct_answer"] = letters[order.index(correct_index)]


def precheck_text_question(q, existing):
    if not isinstance(q, dict):
        return False, "not an object"
    for field in REQUIRED_FIELDS:
        value = q.get(field)
        if value is None or (isinstance(value, str) and not value.strip()):
            return False, f"missing/empty '{field}'"
        if field != "correct_answer":
            q[field] = str(value).strip()

    raw_answer = str(q["correct_answer"]).strip().upper()
    m = re.search(r"\b([ABCD])\b", raw_answer)
    answer = m.group(1) if m else raw_answer
    if answer not in ["A", "B", "C", "D"]:
        return False, f"invalid answer '{raw_answer}'"
    q["correct_answer"] = answer

    if len(q["question"]) < 40:
        return False, "question too short / trivial"
    if len(q["explanation"]) < 40:
        return False, "explanation too short"
    if TEXT_REFERS_TO_IMAGE.search(q["question"]):
        return False, "text question refers to a figure that does not exist"

    options_norm = {re.sub(r"\s+", " ", q[f"option_{l}"].lower()) for l in "ABCD"}
    if len(options_norm) < 4:
        return False, "options are not distinct"

    for k in IMAGE_FIELDS:
        q.pop(k, None)
    q["question_type"] = "text"
    difficulty_info = assess_text_difficulty(q)
    q["difficulty_category"] = difficulty_info["category"]
    q["reasoning_steps"] = max(
        MT_HARD_MIN_REASONING_STEPS,
        2 if difficulty_info["multi_step"] or difficulty_info["quantitative"] else 1
    )

    # Reject questions that are clearly direct recall. A conceptual question can
    # still pass when it has application/reasoning signals.
    if difficulty_info["category"] == "conceptual_direct":
        return False, "too direct/recall-based for Moderate-to-Hard JEE Main"

    if difficulty_info["score"] < 1:
        return False, "insufficient reasoning for Moderate-to-Hard target"

    if is_duplicate(q, existing):
        return False, "duplicate of an earlier question"
    return True, "ok"


# ---------- KG -> LLM prompt ----------

def _format_kg_source(source, number):
    opts = source["options"]
    return (
        f"SOURCE QUESTION {number}\n"
        f"KG ID: {source['kg_question_id']}\n"
        f"Topic: {source['topic']}\n"
        f"Subtopic: {source['subtopic']}\n"
        f"Difficulty: {source['difficulty']}\n"
        f"Question: {source['question']}\n"
        f"A) {opts[0]}\nB) {opts[1]}\nC) {opts[2]}\nD) {opts[3]}\n"
        f"Source correct answer: {source['correct_answer']}\n"
    )


def build_text_prompt(batch_number, sources, avoid_stems):
    source_text = "\n".join(_format_kg_source(s, i + 1) for i, s in enumerate(sources))
    avoid_text = ""
    if avoid_stems:
        avoid_text = (
            "ALREADY GENERATED - do NOT repeat or closely rephrase these:\n"
            + "\n".join("- " + s for s in avoid_stems[-30:]) + "\n"
        )

    return f"""
You are generating JEE Main Chemistry MCQs in Batch {batch_number}.

USER REQUIREMENTS
Exam: JEE Main
Subject: Chemistry
Topic: {TOPIC}
Requested difficulty: {DIFFICULTY}
Requested question type: {QUESTION_TYPE}

KNOWLEDGE-GRAPH RULE
The SOURCE QUESTIONS below were retrieved from the Chemistry Knowledge Graph.
They are the factual/conceptual source for your output. You MUST use them.
Do NOT invent an unrelated topic.

For each source, create ONE NEW question that tests the same underlying chemistry
concept. You may change the wording, numerical values, reaction framing, distractors,
or reasoning path, but preserve scientific correctness and the underlying concept.
Do NOT simply copy the source question.
Do NOT mention the Knowledge Graph or the source in the final question.
Do NOT use images, diagrams, figures, or phrases such as "structure shown".

DIFFICULTY CONTROL
{DIFFICULTY_GUIDE}

IMPORTANT: The requested "Moderate" setting means MODERATE-TO-HARD JEE MAIN.
Do not satisfy the request by merely making the wording longer. Increase the actual
reasoning required. For the text questions in this batch:
- At least one question in every batch should be quantitative or multi-step whenever possible.
- Across the complete text set, target roughly 45-55% numerical/equation/formula-based questions.
- Target roughly 25-35% multi-step/application/data/graph/reaction-sequence questions.
- Keep direct one-line factual/recalI questions below 20%. Prefer transforming a source
  into a calculation, comparison, numerical application, reaction sequence, equilibrium
  calculation, kinetics calculation, or statement-analysis problem when scientifically valid.
- A good question should normally require at least TWO reasoning operations, such as
  identify the governing equation -> substitute -> calculate; or identify the reaction
  sequence -> determine the intermediate -> calculate the final quantity.
- Use realistic JEE Main values and close distractors based on common calculation mistakes.
- Do NOT make every question numerical. Retain conceptual questions, but they must involve
  application, comparison of competing effects, exception/trend reasoning, or interpretation
  rather than simple memorisation.
- Never introduce fake data or an equation that does not follow from the source concept.

QUALITY RULES
- exactly one correct option
- four distinct options
- distractors should be plausible and based on common mistakes
- calculations must be internally consistent
- explanations must show the actual reasoning or calculation, preferably step-by-step
- output must be text-only and independently answerable
- do not change the underlying scientific answer merely to make a new question
- do not use definitions, common names, simple naming, or single-fact recall as the main task
- do not use "which of the following is" unless the options can only be distinguished after
  doing meaningful chemical reasoning

{avoid_text}

SOURCE QUESTIONS FROM KNOWLEDGE GRAPH
{source_text}

OUTPUT
Return ONLY this JSON object:
{{
  "questions": [
    {{
      "question": "...",
      "option_A": "...",
      "option_B": "...",
      "option_C": "...",
      "option_D": "...",
      "correct_answer": "A",
      "explanation": "...",
      "question_type": "text"
    }}
  ]
}}

Return exactly {len(sources)} questions, in the same order as the source questions.
No Markdown. No ```json. No commentary. No <think>.
"""

# ============================================================
# SOURCE SELECTION FOR HARDER TEXT QUESTIONS
# ============================================================

def source_hardness_score(source):
    text = source["question"] + " " + " ".join(source.get("options", []))
    score = 0
    if re.search(r"\d|=|%|mol|M|pH|Ka|Kb|Ksp|rate|energy|volume|mass|pressure|temperature", text, re.I):
        score += 2
    if re.search(r"calculate|determine|find|evaluate|ratio|given|mixture|equilibrium|reaction|successive|graph|data", text, re.I):
        score += 2
    if len(text) > 220:
        score += 1
    if re.search(r"which of the following is|what is the definition|common name", text, re.I):
        score -= 2
    return score


def select_text_sources(available_sources, count, existing):
    """Prefer quantitative/application sources while retaining topic diversity."""
    if count <= 0:
        return []
    ranked = sorted(available_sources, key=source_hardness_score, reverse=True)
    top_n = min(len(ranked), max(count * 5, 20))
    candidate_pool = ranked[:top_n]
    random.shuffle(candidate_pool)

    # Bias the pool toward harder source questions rather than always taking
    # completely random KG nodes.
    weights = [max(1, source_hardness_score(s) + 2) for s in candidate_pool]
    chosen = []
    remaining = list(candidate_pool)
    while remaining and len(chosen) < count:
        total = sum(weights[:len(remaining)])
        r = random.uniform(0, total)
        acc = 0
        pick_idx = 0
        for idx, w in enumerate(weights[:len(remaining)]):
            acc += w
            if r <= acc:
                pick_idx = idx
                break
        chosen.append(remaining.pop(pick_idx))
        weights.pop(pick_idx)
    return chosen


# ============================================================
# BATCH PLAN
# ============================================================

def create_batch_plan(total_questions, total_text, total_image, batch_size):
    plan = []
    remaining_total = total_questions
    remaining_text = total_text
    remaining_image = total_image
    batch_number = 1

    while remaining_total > 0:
        current_size = min(batch_size, remaining_total)

        if remaining_text > remaining_image:
            batch_text = min(3, current_size, remaining_text)
        elif remaining_image > remaining_text:
            batch_text = min(2, current_size, remaining_text)
        else:
            batch_text = min(3, current_size, remaining_text)

        batch_image = current_size - batch_text

        if batch_image > remaining_image:
            batch_image = remaining_image
            batch_text = current_size - batch_image

        if batch_text > remaining_text:
            batch_text = remaining_text
            batch_image = current_size - batch_text

        plan.append({"batch": batch_number, "total": current_size,
                     "text": batch_text, "image": batch_image})

        remaining_total -= current_size
        remaining_text -= batch_text
        remaining_image -= batch_image
        batch_number += 1

    return plan


def slot_types(batch_total, batch_text, batch_image):
    types = []
    text_remaining, image_remaining = batch_text, batch_image
    for i in range(1, batch_total + 1):
        if text_remaining > 0 and image_remaining > 0:
            if i % 2 == 1:
                types.append("text")
                text_remaining -= 1
            else:
                types.append("image")
                image_remaining -= 1
        elif text_remaining > 0:
            types.append("text")
            text_remaining -= 1
        else:
            types.append("image")
            image_remaining -= 1
    return types


batch_plan = create_batch_plan(QUESTION_COUNT, TEXT_COUNT, IMAGE_COUNT, BATCH_SIZE)

print("\n" + "=" * 70)
print("BATCH PLAN")
print("=" * 70)
for b in batch_plan:
    print(f"Batch {b['batch']}: {b['total']} total | {b['text']} text | {b['image']} image")
print("=" * 70)


# ============================================================
# GENERATE ALL BATCHES
# ============================================================

all_questions = []
overall_start = time.time()

print("\n" + "=" * 70)
print("STARTING GENERATION")
print("=" * 70)

for batch_info in batch_plan:

    batch_number = batch_info["batch"]
    slots = slot_types(batch_info["total"], batch_info["text"], batch_info["image"])
    filled = [None] * len(slots)

    print("\n" + "#" * 70)
    print(f"BATCH {batch_number} / {len(batch_plan)}  |  slots: {slots}")
    print("#" * 70)

    # ---- 1. IMAGE slots: built by code, always succeed ----
    for i, t in enumerate(slots):
        if t == "image":
            q = build_image_question(IMAGE_POOL)
            filled[i] = q
            print(f"  slot {i + 1} (image): built [{q.get('image_family')}] "
                  f"- answer {q['correct_answer']} (computed)")

    # ---- 2. TEXT slots: KG source -> Qwen3-8B -> validated question ----
    attempt = 0

    while True:
        missing = [i for i, x in enumerate(filled) if x is None]
        if not missing:
            break

        attempt += 1
        if attempt > MAX_BATCH_ATTEMPTS:
            raise RuntimeError(
                f"Batch {batch_number}: could not fill {len(missing)} text slot(s) "
                f"after {MAX_BATCH_ATTEMPTS} attempts.")

        existing = all_questions + [x for x in filled if x is not None]
        avoid_stems = [str(q["question"])[:110] for q in existing]

        # Select unused KG sources for the missing slots.
        used_kg_ids = {q.get("kg_source_id") for q in existing if q.get("kg_source_id")}
        available_sources = [s for s in KG_TEXT_SOURCES if s["kg_question_id"] not in used_kg_ids]
        if len(available_sources) < len(missing):
            # Reuse is allowed only after the KG pool is exhausted; the LLM still has
            # to create a new question and duplicate checks will reject close copies.
            available_sources = KG_TEXT_SOURCES

        sources = select_text_sources(available_sources, len(missing), existing)
        if len(sources) != len(missing):
            raise RuntimeError("Not enough text-only Knowledge Graph source questions.")

        print(f"\n--- Batch {batch_number} attempt {attempt}: need {len(missing)} text question(s)")
        print("    Sources:", [s["kg_question_id"] for s in sources])

        prompt = build_text_prompt(batch_number, sources, avoid_stems)

        try:
            raw = run_generation(
                prompt, tag=f"batch_{batch_number}_attempt_{attempt}",
                temperature=TEMPERATURE if attempt == 1 else RETRY_TEMPERATURE)
        except torch.cuda.OutOfMemoryError:
            print("CUDA OUT OF MEMORY - retrying.")
            clear_cuda_memory()
            continue
        except Exception as e:
            print("ERROR during generation:", type(e).__name__, e)
            traceback.print_exc()
            clear_cuda_memory()
            continue

        data = extract_json(raw)
        questions = data.get("questions") if isinstance(data, dict) else None

        if not isinstance(questions, list) or not questions:
            print("Could not extract a usable 'questions' list.")
            continue

        candidates = []   # (slot_index, question, source)
        for j, q in enumerate(questions[:len(missing)]):
            slot_index = missing[j]
            source = sources[j]
            ok, msg = precheck_text_question(
                q, existing + [c[1] for c in candidates])
            if ok:
                q["source"] = "Knowledge Graph -> Qwen3-8B"
                q["kg_source_id"] = source["kg_question_id"]
                q["kg_source_question"] = source["question"]
                q["kg_source_topic"] = source["topic"]
                q["kg_source_subtopic"] = source["subtopic"]
                q["kg_source_difficulty"] = source["difficulty"]
                candidates.append((slot_index, q, source))
            else:
                print(f"  slot {slot_index + 1}: rejected - {msg}")

        # independent self-check
        verdicts = [None] * len(candidates)
        if VERIFY_TEXT and candidates:
            verdicts = verify_questions([c[1] for c in candidates])

        strict = attempt <= STRICT_VERIFY_ATTEMPTS
        accepted = 0

        for (slot_index, q, source), verdict in zip(candidates, verdicts):
            if not VERIFY_TEXT:
                q["verification"] = "skipped"
            elif verdict is None:
                q["verification"] = "inconclusive"
            elif verdict == q["correct_answer"]:
                q["verification"] = "agreed"
            else:
                if strict:
                    print(f"  slot {slot_index + 1}: rejected - independent solve chose "
                          f"{verdict}, key says {q['correct_answer']}")
                    continue
                q["verification"] = "disputed"
                q["solver_answer"] = verdict

            shuffle_options(q)
            filled[slot_index] = q
            accepted += 1
            print(f"  slot {slot_index + 1}: accepted ({q['verification']}) "
                  f"from KG {source['kg_question_id']}")

        print(f"Accepted {accepted}/{len(missing)} in this attempt.")

    all_questions.extend(filled)
    print(f"\nBatch {batch_number} complete ({len(all_questions)} questions so far).")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump({"questions": all_questions}, f, indent=4, ensure_ascii=False)


# ============================================================
# FINAL CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL GENERATION CHECK")
print("=" * 70)


def count_image_questions(questions):
    return sum(1 for q in questions if q.get("question_type") == "image")


final_image_count = count_image_questions(all_questions)
final_text_count = len(all_questions) - final_image_count

print("Expected questions :", QUESTION_COUNT)
print("Generated questions:", len(all_questions))
print("Text questions     :", final_text_count, f"(requested {TEXT_COUNT})")
print("Image questions    :", final_image_count, f"(requested {IMAGE_COUNT})")

if len(all_questions) != QUESTION_COUNT:
    raise RuntimeError(f"Expected {QUESTION_COUNT} questions, got {len(all_questions)}.")
if final_text_count != TEXT_COUNT or final_image_count != IMAGE_COUNT:
    raise RuntimeError("Text/image counts do not match the request.")

ver_counts = Counter(q.get("verification", "?") for q in all_questions)
print("Verification       :", dict(ver_counts))

# Difficulty/style report. This is deliberately printed so you can see whether
# the generator actually produced the requested Moderate-to-Hard mix.
style_counts = Counter(q.get("difficulty_category", "computed_image") for q in all_questions)
print("Difficulty categories:", dict(style_counts))
if final_text_count:
    text_style = Counter(q.get("difficulty_category", "unknown")
                         for q in all_questions if q.get("question_type") == "text")
    print("Text style categories:", dict(text_style))

final_output = {"questions": all_questions}


# ============================================================
# SAVE JSON + PICKLE
# ============================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=4, ensure_ascii=False)
print("\nJSON saved   :", OUTPUT_FILE)

with open(PICKLE_OUTPUT_FILE, "wb") as f:
    pickle.dump(final_output, f)
print("Pickle saved :", PICKLE_OUTPUT_FILE)


# ============================================================
# EXPORT: HTML
# ============================================================

def image_to_data_uri(path):
    try:
        with open(path, "rb") as f:
            return "data:image/png;base64," + base64.b64encode(f.read()).decode("utf-8")
    except Exception:
        return None


def review_note(q):
    v = q.get("verification")
    if v == "disputed":
        return (f"Needs review: independent self-check chose {q.get('solver_answer')} "
                f"but the key says {q['correct_answer']}.")
    if v == "inconclusive":
        return "Needs review: self-check could not reach a clear answer."
    return ""


def export_html(questions, output_path):
    esc = html.escape
    parts = [
        "<!DOCTYPE html><html><head><meta charset='utf-8'>",
        "<title>JEE Main Chemistry Questions</title>",
        "<style>",
        "body{font-family:Arial,Helvetica,sans-serif;max-width:900px;margin:auto;padding:20px;line-height:1.5}",
        ".q{border:1px solid #ccc;border-radius:8px;padding:16px;margin:20px 0}",
        ".opt{margin:4px 0 4px 16px}",
        ".ans{margin-top:10px;padding:8px;background:#eef7ee;border-radius:6px}",
        ".warn{margin-top:8px;padding:8px;background:#fff3cd;border-radius:6px}",
        "img{max-width:100%;border:1px solid #ddd;border-radius:6px}",
        "</style></head><body>",
        "<h1>JEE Main Chemistry Question Bank</h1>",
        f"<p>Topic: {esc(str(TOPIC))} | Difficulty: {esc(str(DIFFICULTY))} | "
        f"Total: {len(questions)}</p>"
    ]

    for i, q in enumerate(questions, start=1):
        parts.append("<div class='q'>")
        parts.append(f"<h3>Q{i}. [{esc(q.get('question_type', '').upper())}]</h3>")
        parts.append(f"<p>{esc(str(q['question']))}</p>")

        if q.get("question_type") == "image":
            uri = image_to_data_uri(q["generated_image"]) if q.get("generated_image") else None
            if uri:
                parts.append(f"<img src='{uri}' alt='Question {i} figure'>")

        for letter in "ABCD":
            parts.append(f"<div class='opt'><b>({letter})</b> {esc(str(q[f'option_{letter}']))}</div>")

        parts.append(
            f"<div class='ans'><b>Correct answer:</b> {esc(str(q['correct_answer']))}<br>"
            f"<b>Explanation:</b> {esc(str(q['explanation']))}</div>")

        note = review_note(q)
        if note:
            parts.append(f"<div class='warn'>{esc(note)}</div>")

        parts.append("</div>")

    parts.append("</body></html>")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(parts))


try:
    export_html(all_questions, HTML_OUTPUT_FILE)
    print("HTML saved   :", HTML_OUTPUT_FILE)
except Exception as e:
    print("HTML export failed:", e)
    traceback.print_exc()


# ============================================================
# EXPORT: WORD
# ============================================================

def export_docx(questions, output_path):
    from docx import Document
    from docx.shared import Inches

    doc = Document()
    doc.add_heading("JEE Main Chemistry Question Bank", level=0)
    doc.add_paragraph(f"Topic: {TOPIC} | Difficulty: {DIFFICULTY} | Total: {len(questions)}")

    for i, q in enumerate(questions, start=1):
        doc.add_heading(f"Q{i}. [{q.get('question_type', '').upper()}]", level=2)
        doc.add_paragraph(str(q["question"]))

        if q.get("question_type") == "image":
            path = q.get("generated_image")
            if path and os.path.exists(path):
                doc.add_picture(path, width=Inches(4.5))

        for letter in "ABCD":
            doc.add_paragraph(f"({letter}) {q[f'option_{letter}']}")

        p = doc.add_paragraph()
        p.add_run("Correct answer: ").bold = True
        p.add_run(str(q["correct_answer"]))

        p = doc.add_paragraph()
        p.add_run("Explanation: ").bold = True
        p.add_run(str(q["explanation"]))

        note = review_note(q)
        if note:
            p = doc.add_paragraph()
            p.add_run(note).italic = True

    doc.save(output_path)


try:
    export_docx(all_questions, WORD_OUTPUT_FILE)
    print("Word saved   :", WORD_OUTPUT_FILE)
except Exception as e:
    print("Word export failed:", e)
    traceback.print_exc()


# ============================================================
# FINAL REPORT
# ============================================================

print("\n" + "=" * 70)
print("AGENT 4 v2 COMPLETED")
print("=" * 70)
print("Total questions :", len(all_questions))
print("Text / Image    :", final_text_count, "/", final_image_count)
print("Verification    :", dict(ver_counts))
print("Total time      :", round((time.time() - overall_start) / 60, 2), "minutes")
print("Answer letters  :", {l: sum(1 for q in all_questions if q["correct_answer"] == l) for l in "ABCD"})
print("=" * 70)

print("\nQUESTION SUMMARY")
print("-" * 70)
for index, question in enumerate(all_questions, start=1):
    flag = "" if question.get("verification") in ("agreed", "computed", "skipped") else "  [REVIEW]"
    print(f"{index:02d}. [{question.get('question_type', '?').upper()}] "
          f"{question['question'][:100]}{flag}")

print("\nImage-question answers are computed by code. Text questions are generated "
      "by Qwen3-8B from Knowledge Graph source questions and self-checked; "
      "review any flagged [REVIEW] items and spot-check the rest.")

try:
    from google.colab import files
    for path in [OUTPUT_FILE, HTML_OUTPUT_FILE, WORD_OUTPUT_FILE, PICKLE_OUTPUT_FILE]:
        if os.path.exists(path):
            files.download(path)
except Exception:
    print("Auto-download skipped.")

Transformers : 5.19.0
Accelerate   : 1.15.0
BitsAndBytes : 0.50.2
PyTorch      : 2.11.0+cu128
GPU: Tesla T4
GPU Memory at startup: Allocated=0.00 GB | Reserved=0.00 GB | Total=14.56 GB

Agent 2 Requirements:
{
    "Number": 200,
    "Exam": "JEE Main",
    "Subject": "Chemistry",
    "Topic": "ALL",
    "Concept": null,
    "Difficulty": "Moderate",
    "Type": "MCQ"
}

QUESTION GENERATION MODE
1. text
2. image
3. mixed



Question type (text/image/mixed):  mixed

How many of 200 questions should be image-based?  40



GENERATION CONFIGURATION
Total questions : 200
Mode            : mixed
Text / Image    : 160 / 40
Topic           : ALL
Difficulty      : Moderate
Question type   : MCQ
Self-check      : True

TEXT SOURCE: KNOWLEDGE GRAPH
KG path              : /kaggle/working/chemistry_knowledge_graph.pkl
Text-only KG sources : 344
Topic filter         : ALL
Difficulty preference: Moderate
LLM                   : Qwen/Qwen3-8B

Loading tokenizer for KG -> LLM generation...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
import os
import shutil
from IPython.display import FileLink

print("--- Step 1: Collecting target files and specific image folders ---")
working_dir = '/kaggle/working/'
output_dir = '/kaggle/working/clean_staging'

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

# Define exact extensions and folders you want to include
target_extensions = ('.pkp', '.docx', '.doc', '.html', '.htm', '.json')
allowed_folders = ['generated_question_images', 'chemistry_images']

for item in os.listdir(working_dir):
    src_path = os.path.join(working_dir, item)
    
    # Copy target files
    if os.path.isfile(src_path) and item.lower().endswith(target_extensions):
        shutil.copy(src_path, output_dir)
        
    # Copy only the specific image folders (ignoring system/cache folders)
    elif os.path.isdir(src_path) and item in allowed_folders:
        dest_path = os.path.join(output_dir, item)
        shutil.copytree(src_path, dest_path)

print("--- Step 2: Creating zip archive ---")
zip_name = '/kaggle/working/chemistry_outputs'
if os.path.exists(f"{zip_name}.zip"):
    os.remove(f"{zip_name}.zip")

shutil.make_archive(zip_name, 'zip', output_dir)
size_mb = os.path.getsize(f"{zip_name}.zip") / (1024 * 1024)
print(f"Zip created successfully! Size: {size_mb:.2f} MB")

print("\n--- Step 3: Generating fast download link ---")
display(FileLink('chemistry_outputs.zip', result_html_prefix="Click here to download your zip file: "))

--- Step 1: Collecting target files and specific image folders ---
--- Step 2: Creating zip archive ---
Zip created successfully! Size: 56.03 MB

--- Step 3: Generating fast download link ---


/kaggle/working/chemistry_outputs.zip